# Preprocessing

Here we start witht the preprocessing stage of data analysis.

We work with the copy of cumulative_raw_data.csv called optimized_dataset.csv that we will edit

Couple of issues to fix here:
- combining the same columns
- combining the same row inputs (translating them into english)

In [2]:
import os
import pandas as pd
import numpy as np

In [3]:
df_bosnian = pd.read_csv('../data/Bosnian_data.csv')
df_turkish = pd.read_csv('../data/Turkish_data.csv')
df_english = pd.read_csv('../data/English_data.csv')

Since we tailored the surveys to take same questions in different languages, we can just connect the datasets into one by the index of the column. That is the simplest way. 




In [4]:
# combine the datasets into one cumulative_data.csv

english_columns = df_english.columns

df_bosnian_copy = df_bosnian.copy()
df_bosnian_copy.columns = english_columns

df_turkish_copy = df_turkish.copy()
df_turkish_copy.columns = english_columns

cumulative_df = pd.concat([df_english, df_bosnian_copy, df_turkish_copy], ignore_index=True)
cumulative_df.to_csv('../data/optimized_data1.csv', index=False)

print('Cumulative shape:', cumulative_df.shape)
cumulative_df.head()

Cumulative shape: (177, 100)


,Timestamp,Do you agree that your anonymised answers can be used for research and model development for this endometriosis/PCOS project?,What year were you born?,What is your country of residence?,What is your height (in cm)?,What is your weight (in kg)?,How often do you workout or do intentional physical activity?,How often do you eat fast food or high processed food?,Have you been diagnosed with endometriosis?,Have you been diagnosed you with polycystic ovary syndrome (PCOS)?,...,Higher protein diet,Lower‑carb diet,Keto diet,Anti‑inflammatory diet,Reduced sugar / sweets,Reduced processed / fast food,Reduced gluten,Reduced dairy,Vegetarian or vegan,Intermittent fasting / time‑restricted eating
0,3/23/2026 13:27:25,"Yes, I agree",2003,Bosnia and Herzegovina,168,55,3-4 times per week,Rarely or never,"No, and I do not think I have it","No, and I do not think I have it",...,5.0,7.0,0.0,6.0,10.0,9.0,0.0,0.0,0.0,5.0
1,3/23/2026 15:37:41,"Yes, I agree",2002,Bosnia,157,41,1-2 times per week,5+ times per week,"No, but I suspect I have it","No, and I do not think I have it",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3/23/2026 16:56:11,"Yes, I agree",2006,qatar,158,53,0 times per week,Rarely or never,"No, and I do not think I have it","No, but I suspect I have it",...,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0
3,3/23/2026 17:15:06,"Yes, I agree",2003,Germany,178,60,1-2 times per week,1-2 times per week,"No, but a doctor suspects it","No, but I suspect I have it",...,0.0,0.0,0.0,7.0,6.0,6.0,0.0,0.0,0.0,0.0
4,3/23/2026 17:15:13,"Yes, I agree",2003,Germany,178,60,1-2 times per week,1-2 times per week,"No, but a doctor suspects it","No, but I suspect I have it",...,0.0,0.0,0.0,7.0,6.0,6.0,0.0,0.0,0.0,0.0


We successfully concateneted.

Now we need to map same row values from different languages.
There are a few ways this will be done, based on the values:
- Skipped for numerical values
- For certain yes/no questions, it will be immediately mapped to 1 and 0

In [5]:
# list columns with 2 values

# Print columns with only two unique non-null values and show their unique values
for col in df_english.columns:
    unique_vals = df_english[col].dropna().unique()
    if len(unique_vals) == 2:
        print(f"{col}: {unique_vals}")

Do you agree that your anonymised answers can be used for research and model development for this endometriosis/PCOS project?: ['Yes, I agree' 'No, I do not agree']
Are you currently pregnant?: ['No' 'Yes']
Berberine: [0. 5.]


In [6]:
# setting all country values to be same
# eg fixing bosnia vs bih vs BiH and so on...

# print unique values in country column

print('Unique values in country column:', cumulative_df['What is your country of residence?'].unique())


Unique values in country column: ['Bosnia and Herzegovina' 'Bosnia' 'qatar' 'Germany' 'BiH' 'Portugal'
 'Italy' nan 'Egypt/Bosnia' 'Azerbaijan' 'Bosnia and Herzegovina '
 'Turkey ' 'Austria' 'Germany ' 'Hrvatski ' 'Kenya' 'Croatia' 'Kenya '
 'Egypt ' 'UAE' 'Egypt' 'Bosna' 'Srbija' 'Bosna i Hercegovina'
 'Bosna i Hercegovina ' 'Bosna i Herecegovina' 'Srbiji' 'Nemacka' 'BIH'
 'Austrija' 'Hrvatska ' 'Hrvatska' 'Srbija ' 'bih' 'Njemacka' 'SRBIJA '
 'Bih' 'USA ' 'Hrvstska' 'u Srbiji' 'Republika Srbija' 'Bosna '
 'Slovenija' 'Bosni i Hercegovini' 'BiH ' 'Hr' 'Bosna I Hercegovina '
 'Republika Srpska' 'Türkiye' 'Almanya']


In [7]:
def standardize_country(country):
    if pd.isnull(country):
        return country
    c = country.strip().lower()
    # Bosnia and Herzegovina
    if any(x in c for x in ['bos', 'bih']):
        return 'Bosnia and Herzegovina'
    # Serbia
    if any(x in c for x in ['srb', 'serb']):
        return 'Serbia'
    # Croatia
    if any(x in c for x in ['hrv', 'croa', 'hr']):
        return 'Croatia'
    # Germany
    if any(x in c for x in ['ger', 'njem', 'nem', 'alm']):
        return 'Germany'
    # Turkey
    if any(x in c for x in ['turk', 'tür', 'tur']):
        return 'Turkey'
    # Austria
    if any(x in c for x in ['aust', 'aus']):
        return 'Austria'
    # Qatar
    if any(x in c for x in ['qat', 'qatar']):
        return 'Qatar'
    # Slovenia
    if any(x in c for x in ['slo', 'sloven']):
        return 'Slovenia' 
    return country.strip()

cumulative_df['What is your country of residence?'] = cumulative_df['What is your country of residence?'].apply(standardize_country)

print('Standardized values:', cumulative_df['What is your country of residence?'].unique())

Standardized values: ['Bosnia and Herzegovina' 'Qatar' 'Germany' 'Portugal' 'Italy' nan
 'Azerbaijan' 'Turkey' 'Austria' 'Croatia' 'Kenya' 'Egypt' 'UAE' 'Serbia'
 'USA' 'Slovenia' 'Republika Srpska']


In [8]:
cumulative_df.head(70)

,Timestamp,Do you agree that your anonymised answers can be used for research and model development for this endometriosis/PCOS project?,What year were you born?,What is your country of residence?,What is your height (in cm)?,What is your weight (in kg)?,How often do you workout or do intentional physical activity?,How often do you eat fast food or high processed food?,Have you been diagnosed with endometriosis?,Have you been diagnosed you with polycystic ovary syndrome (PCOS)?,...,Higher protein diet,Lower‑carb diet,Keto diet,Anti‑inflammatory diet,Reduced sugar / sweets,Reduced processed / fast food,Reduced gluten,Reduced dairy,Vegetarian or vegan,Intermittent fasting / time‑restricted eating
0,3/23/2026 13:27:25,"Yes, I agree",2003,Bosnia and Herzegovina,168,55,3-4 times per week,Rarely or never,"No, and I do not think I have it","No, and I do not think I have it",...,5.0,7.0,0.0,6.0,10.0,9.0,0.0,0.0,0.0,5.0
1,3/23/2026 15:37:41,"Yes, I agree",2002,Bosnia and Herzegovina,157,41,1-2 times per week,5+ times per week,"No, but I suspect I have it","No, and I do not think I have it",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3/23/2026 16:56:11,"Yes, I agree",2006,Qatar,158,53,0 times per week,Rarely or never,"No, and I do not think I have it","No, but I suspect I have it",...,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0
3,3/23/2026 17:15:06,"Yes, I agree",2003,Germany,178,60,1-2 times per week,1-2 times per week,"No, but a doctor suspects it","No, but I suspect I have it",...,0.0,0.0,0.0,7.0,6.0,6.0,0.0,0.0,0.0,0.0
4,3/23/2026 17:15:13,"Yes, I agree",2003,Germany,178,60,1-2 times per week,1-2 times per week,"No, but a doctor suspects it","No, but I suspect I have it",...,0.0,0.0,0.0,7.0,6.0,6.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,3/23/2026 19:58:38,"Da, slažem se",2003,Bosnia and Herzegovina,170,80,1-2 puta u sedmici,Rijetko ili nikako,"Ne, ne mislim da je imam",Nisam sigurna,...,3.0,0.0,0.0,0.0,0.0,0.0,2.0,3.0,0.0,1.0
66,3/23/2026 20:06:48,"Da, slažem se",2000,Bosnia and Herzegovina,171,77,1-2 puta u sedmici,Rijetko ili nikako,"Ne, ali ljekar sumnja na to","Ne, ne mislim da je imam",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
67,3/23/2026 20:07:24,"Da, slažem se",2004,Bosnia and Herzegovina,172,90,0 puta u sedmici,3-4 puta u sedmici,"Ne, ne mislim da je imam","Ne, ne mislim da je imam",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
68,3/23/2026 20:09:37,"Da, slažem se",1984,Croatia,168,61,0 puta u sedmici,Rijetko ili nikako,"Da, zvanično dijagnosticirana od strane ljekara","Ne, ali ja mislim da imam",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
# add it to the cumulative dataset
cumulative_df.to_csv('../data/cumulative_data.csv', index=False)

Since some data is redundant for the model trianing and we don't want it to cause noise, we will make a duplicate of the dataset, one will be kept with all values (such as country of residence) for statistics and research purposes, and another cleaned up one will be used for pure relevant data for model training.



eg code 

Full dataset for research/statistics
research_df = cumulative_df.copy()

Training dataset: drop rows with missing country (or any other cleaning)
training_df = cumulative_df.dropna(subset=['What is your country of residence?'])

Save both datasets
research_df.to_csv('../data/cumulative_data_research.csv', index=False)
training_df.to_csv('../data/cumulative_data_training.csv', index=False)

In [10]:
import re

def clean_year_of_birth(value):
    if pd.isnull(value):
        return np.nan
    value = str(value).strip()
    value = re.sub(r'^\D+', '', value)
    if value.isdigit():
        num = int(value)
        if num < 100:
            return 1900 + num
        return num
    return np.nan

cumulative_df['What year were you born?'] = cumulative_df['What year were you born?'].apply(clean_year_of_birth)
cumulative_df['What year were you born?'] = cumulative_df['What year were you born?'].astype('Int64')
print(cumulative_df['What year were you born?'].unique())

<IntegerArray>
[2003, 2002, 2006, 1999, 1995, <NA>, 2005, 2004, 1991, 1988, 2001, 1994, 1996,
 1998, 1997, 2000, 1992, 1990, 1979, 1982, 1985, 1984, 1980, 1987, 1993, 1976,
 1981, 1989, 1902, 1977, 1986, 1975, 1978]
Length: 33, dtype: Int64


In [11]:
print(cumulative_df['What year were you born?'].head(50))

0     2003
1     2002
2     2006
3     2003
4     2003
5     1999
6     1995
7     2002
8     2002
9     <NA>
10    <NA>
11    2005
12    2002
13    2004
14    2003
15    2002
16    2002
17    2005
18    2002
19    2006
20    1991
21    2003
22    2006
23    2005
24    2002
25    1988
26    1988
27    2001
28    1999
29    2002
30    1994
31    1988
32    2004
33    1999
34    2004
35    2004
36    1996
37    2003
38    2003
39    2003
40    2002
41    2001
42    1996
43    1999
44    1998
45    2003
46    2002
47    2003
48    1997
49    2002
Name: What year were you born?, dtype: Int64


In [12]:
# print column names from culmulative dataset
print(cumulative_df.columns)

Index(['Timestamp',
       'Do you agree that your anonymised answers can be used for research and model development for this endometriosis/PCOS project?',
       'What year were you born?', 'What is your country of residence?',
       'What is your height (in cm)?', 'What is your weight (in kg)?',
       'How often do you workout or do intentional physical activity?',
       'How often do you eat fast food or high processed food?',
       'Have you been diagnosed with endometriosis?',
       'Have you been diagnosed you with polycystic ovary syndrome (PCOS)?',
       'Which race or ethnic group(s) do you identify with? (Optional, select all that apply)',
       'At what age did you have your first period?',
       'How would you describe your menstrual cycle pattern in the last year?',
       'How long is your typical menstrual cycle? (Number of days from Day 1 to the next Day 1)',
       'How many days do you usually bleed during your period?',
       'Do you experience heavy or extr

### Remove the empty rows without accepted permission

In [13]:
# Remove rows where user did not agree to research usage
cumulative_df = cumulative_df[cumulative_df['Do you agree that your anonymised answers can be used for research and model development for this endometriosis/PCOS project?'] != "No, I do not agree"]


In [14]:
for col in cumulative_df.columns:
    print(f"Unique values in '{col}':")
    print(cumulative_df[col].unique())
    print("-" * 40)

Unique values in 'Timestamp':
['3/23/2026 13:27:25' '3/23/2026 15:37:41' '3/23/2026 16:56:11'
 '3/23/2026 17:15:06' '3/23/2026 17:15:13' '3/23/2026 17:24:53'
 '3/23/2026 17:56:11' '3/23/2026 17:57:38' '3/23/2026 18:01:42'
 '3/23/2026 18:22:07' '3/23/2026 18:54:50' '3/23/2026 18:59:30'
 '3/23/2026 19:12:16' '3/23/2026 19:24:03' '3/23/2026 19:25:16'
 '3/23/2026 19:29:20' '3/23/2026 19:37:08' '3/23/2026 19:40:52'
 '3/23/2026 20:04:25' '3/23/2026 20:28:28' '3/23/2026 21:01:35'
 '3/23/2026 21:41:09' '3/23/2026 21:44:39' '3/24/2026 0:49:31'
 '3/24/2026 7:12:22' '3/24/2026 8:54:25' '3/24/2026 15:35:43'
 '3/24/2026 21:44:32' '3/24/2026 21:53:08' '3/24/2026 22:45:36'
 '3/26/2026 12:51:55' '3/26/2026 23:07:52' '3/27/2026 9:02:31'
 '3/27/2026 16:27:10' '4/4/2026 16:13:53' '4/12/2026 14:17:10'
 '3/23/2026 16:42:32' '3/23/2026 16:51:49' '3/23/2026 17:11:40'
 '3/23/2026 17:17:49' '3/23/2026 17:19:48' '3/23/2026 17:24:07'
 '3/23/2026 17:46:55' '3/23/2026 17:55:52' '3/23/2026 18:09:30'
 '3/23/2026 18:

In [15]:
# fix the missing birth year
median_year = cumulative_df["What year were you born?"].median()
cumulative_df["What year were you born?"] = cumulative_df["What year were you born?"].fillna(median_year)


In [16]:
# standardize all weight values to be in kg and integer type
cumulative_df["What is your weight (in kg)?"] = (
    cumulative_df["What is your weight (in kg)?"]
    .astype(str)
    .str.replace(r"[^\d.,]", "", regex=True)
    .str.replace(",", "", regex=False)
    .replace("", None)
    .astype(float)
    .round(0)
    .astype('Int64')
)
# Remove rows with NA values in weight column
cumulative_df = cumulative_df.dropna(subset=["What is your weight (in kg)?"])
print(cumulative_df['What is your weight (in kg)?'].unique())


<IntegerArray>
[ 55,  41,  53,  60,  50,  52,  97,  63,  56,  49,  80,  79,  58,  68,  69,
  75,  77,  65,  83,  71,  64,  72,  66,  62,  67,  70,  48,  57,  59, 108,
  90,  61,  54,  73,  82, 100,  76,  81,  85,  51, 104,  78, 115,  47,  88]
Length: 45, dtype: Int64


In [17]:
# Standardize 'How often do you workout or do intentional physical activity?' to English categories
def map_workout_frequency(val):
    if pd.isnull(val):
        return None
    val = str(val).strip().lower()
    if val in ['0 times per week', '0 puta u sedmici', 'haftada 0 kez']:
        return '0 times per week'
    if val in ['1-2 times per week', '1-2 puta u sedmici', 'haftada 1–2 kez']:
        return '1-2 times per week'
    if val in ['3-4 times per week', '3-4 puta u sedmici']:
        return '3-4 times per week'
    if val in ['5+ times per week', '5+ puta u sedmici', 'haftada 5 veya daha fazla kez']:
        return '5+ times per week'
    return val
cumulative_df['How often do you workout or do intentional physical activity?'] = cumulative_df['How often do you workout or do intentional physical activity?'].apply(map_workout_frequency)
print(cumulative_df['How often do you workout or do intentional physical activity?'].unique())

['3-4 times per week' '1-2 times per week' '0 times per week'
 '5+ times per week']


In [18]:
# Convert Timestamp to datetime
cumulative_df['Timestamp'] = pd.to_datetime(cumulative_df['Timestamp'], format='%m/%d/%Y %H:%M:%S', errors='coerce')
print(cumulative_df['Timestamp'].head())

0   2026-03-23 13:27:25
1   2026-03-23 15:37:41
2   2026-03-23 16:56:11
3   2026-03-23 17:15:06
4   2026-03-23 17:15:13
Name: Timestamp, dtype: datetime64[ns]


In [19]:
# Standardize 'Do you agree that your anonymised answers can be used...' to English
def map_agreement(val):
    if pd.isnull(val):
        return None
    val = str(val).strip().lower()
    if 'yes' in val or 'da' in val or 'evet' in val:
        return 'Yes'
    return 'No'
cumulative_df['Do you agree that your anonymised answers can be used for research and model development for this endometriosis/PCOS project?'] = cumulative_df['Do you agree that your anonymised answers can be used for research and model development for this endometriosis/PCOS project?'].apply(map_agreement)
print(cumulative_df['Do you agree that your anonymised answers can be used for research and model development for this endometriosis/PCOS project?'].unique())

['Yes']


In [20]:
# Standardize 'How often do you eat fast food...' to English categories
def map_fast_food_frequency(val):
    if pd.isnull(val):
        return None
    val = str(val).strip().lower()
    if val in ['rarely or never', 'rijetko ili nikako', 'hiçbir zaman ve neredeyse hiç']:
        return 'Rarely or never'
    if val in ['1-2 times per week', '1-2 puta u sedmici', 'haftada 1–2 kez']:
        return '1-2 times per week'
    if val in ['3-4 times per week', '3-4 puta u sedmici', 'haftada 3–4 kez']:
        return '3-4 times per week'
    if val in ['5+ times per week', '5+ puta u sedmici']:
        return '5+ times per week'
    return val
cumulative_df['How often do you eat fast food or high processed food?'] = cumulative_df['How often do you eat fast food or high processed food?'].apply(map_fast_food_frequency)
print(cumulative_df['How often do you eat fast food or high processed food?'].unique())

['Rarely or never' '5+ times per week' '1-2 times per week'
 '3-4 times per week']


In [21]:
# Standardize Yes/No/Sometimes/Often columns to English
print(f'Dataset shape before: {cumulative_df.shape}')
print(f'\nProcessing Yes/No/Sometimes/Often columns...')

def map_yes_no_sometimes(val):
    if pd.isnull(val):
        return None
    val = str(val).strip().lower()
    # Yes variants
    if any(x in val for x in ['yes', 'da', 'evet', 'evet.', 'evet ', 'da ', 'yes ', 'evet,', 'da,', 'yes,']):
        return 'Yes'
    # No variants
    if any(x in val for x in ['no', 'ne', 'hayır', 'hayir', 'nikad', 'ne ', 'no ', 'hayır ', 'hayir ', 'no,', 'ne,', 'hayır,', 'hayir,']):
        return 'No'
    # Sometimes/Often variants
    if any(x in val for x in ['sometimes', 'ponekad', 'bazen', 'ara sıra', 'emi tidak', 'ara sira', 'bazen ', 'ponekad ', 'sometimes ', 'bazen,', 'ponekad,', 'sometimes,']):
        return 'Sometimes'
    if any(x in val for x in ['often', 'často', 'cesto', 'sık sık', 'sik sik', 'često', 'cesto', 'often ', 'často ', 'cesto ', 'sık sık ', 'sik sik ', 'often,', 'často,', 'cesto,', 'sık sık,', 'sik sik,']):
        return 'Often'
    if any(x in val for x in ['almost every period', 'skoro svaki ciklus', 'neredeyse her adet', 'neredeyse her dönem', 'neredeyse her periyot', 'skoro svaki ciklus ', 'almost every period ', 'neredeyse her adet ', 'almost every period,', 'skoro svaki ciklus,', 'neredeyse her adet,']):
        return 'Almost every period'
    # If not matched, return 'Other' to ensure English only
    return 'Other'

# Apply to relevant columns
yes_no_cols = [
    'Do you experience heavy or extreme menstrual bleeding...',
    'Do you experience pain or burning during sexual intercourse (dyspareunia)?',
    'Do you experience pain after sexual intercourse?',
    'Do you have painful bowel movements, especially around your period?',
    'Do you have pain or burning when urinating, especially around your period?',
    'Do you experience dizziness or fainting around your period? ',
    'Do you have frequent headaches or migraines related to your cycle? ',
    'Do you often notice blood clots during your period (larger than 1 cm)?',
    'Do your bowel habits (diarrhea/constipation) change frequently throughout the month (IBS‑like symptoms)? '
]

columns_found = []
columns_not_found = []

for col in yes_no_cols:
    if col in cumulative_df.columns:
        cumulative_df[col] = cumulative_df[col].apply(map_yes_no_sometimes)
        columns_found.append(col)
        print(f"Unique values in '{col}':", cumulative_df[col].unique())
    else:
        columns_not_found.append(col)

print(f'✓ Standardized {len(columns_found)} columns')
print(f'⚠ Could not find {len(columns_not_found)} columns in dataset')
print(f'\nDataset shape after: {cumulative_df.shape}')

Dataset shape before: (175, 100)

Processing Yes/No/Sometimes/Often columns...
Unique values in 'Do you experience pain or burning during sexual intercourse (dyspareunia)?': ['No' 'Sometimes' 'Often' None 'Other']
Unique values in 'Do you experience pain after sexual intercourse?': ['No' 'Sometimes' 'Often' None 'Other']
Unique values in 'Do you have painful bowel movements, especially around your period?': ['Sometimes' 'Often' 'No']
Unique values in 'Do you have pain or burning when urinating, especially around your period?': ['No' 'Sometimes' 'Often']
Unique values in 'Do you experience dizziness or fainting around your period? ': ['Sometimes' 'Often' 'No']
Unique values in 'Do you have frequent headaches or migraines related to your cycle? ': ['Sometimes' 'Often' 'No' None]
Unique values in 'Do you often notice blood clots during your period (larger than 1 cm)?': ['Sometimes' 'Almost every period' 'No' 'Often']
Unique values in 'Do your bowel habits (diarrhea/constipation) change fr

In [22]:
# Standardize diagnosis columns (endometriosis and PCOS
def map_diagnosis(val):
    if pd.isnull(val):
        return None
    val = str(val).strip().lower()
    if any(x in val for x in [
        'yes, officially',
        'da, zvanično',
        'evet, resmi',
        'evet, bir doktor tarafından resmi olarak teşhis edildi',
        'yes, officially diagnosed by a doctor',
        'da, zvanično dijagnostikovana od strane doktora'
    ]):
        return 'Yes, officially diagnosed'
    if any(x in val for x in [
        'no, but i suspect',
        'ne, ali ja mislim',
        'hayır, ancak',
        'no, but i suspect i have it',
        'ne, ali ja sumnjam da je imam',
        'hayır, ancak pcos olduğunu düşünüyorum'
    ]):
        return 'No, but suspect'
    if any(x in val for x in [
        'no, and i do not think',
        'ne, ne mislim',
        'hayır ve',
        'no, and i do not think i have it',
        'ne, ne mislim da je imam',
        'hayır ve pcos olduğunu düşünmüyorum'
    ]):
        return 'No, do not think'
    if any(x in val for x in [
        'no, but a doctor suspects',
        'ne, ali ljekar sumnja',
        'ancak bir doktor şüpheleniy',
        'no, but a doctor suspects i have it',
        'ne, ali doktor sumnja da je imam',
        'ancak bir doktor pcos şüpheleniyor'
    ]):
        return 'No, but doctor suspects'
    if any(x in val for x in [
        'not sure',
        'nisam sigurna',
        'emin değilim',
        'not sure / i don\'t remember',
        'nisam sigurna/ ne sjećam se',
        'emin değilim / hatirlamiyorum'
    ]):
        return 'Not sure'
    return 'Other'
cumulative_df['Have you been diagnosed with endometriosis?'] = cumulative_df['Have you been diagnosed with endometriosis?'].apply(map_diagnosis)
cumulative_df['Have you been diagnosed you with polycystic ovary syndrome (PCOS)?'] = cumulative_df['Have you been diagnosed you with polycystic ovary syndrome (PCOS)?'].apply(map_diagnosis)
print('Endometriosis diagnoses:', cumulative_df['Have you been diagnosed with endometriosis?'].unique())
print('PCOS diagnoses:', cumulative_df['Have you been diagnosed you with polycystic ovary syndrome (PCOS)?'].unique())

Endometriosis diagnoses: ['No, do not think' 'No, but suspect' 'No, but doctor suspects'
 'Yes, officially diagnosed' 'Not sure']
PCOS diagnoses: ['No, do not think' 'No, but suspect' 'Yes, officially diagnosed'
 'No, but doctor suspects' 'Not sure']


In [23]:
# Standardize menstrual cycle patterns
def map_cycle_pattern(val):
    if pd.isnull(val):
        return None
    val = str(val).strip().lower()
    if any(x in val for x in [
        'regular',
        'redovan',
        'düzenli',
        'düzenli (yaklaşık her 21–35 günde bir olur)',
        'redovan (javlja se otprilike svakih 21–35 dana)'
    ]):
        return 'Regular'
    if any(x in val for x in [
        'irregular',
        'neredovan',
        'düzensiz',
        'biraz düzensiz',
        'somewhat irregular',
        'djelomično neredovan',
        'biraz düzensiz (bazen çok erken veya çok geç olur)',
        'djelomično neredovan (ponekad veoma rano ili veoma kasno)'
    ]):
        return 'Irregular'
    if any(x in val for x in [
        'continuous hormonal',
        'kontinuiranu hormonsku',
        'kontinu hormonal',
        'koristim kontinuiranu hormonsku kontracepciju i ne krvarim'
    ]):
        return 'Hormonal contraception'
    if any(x in val for x in [
        'no menstruation',
        'nema menstruacije',
        'menstruasyon yok',
        'nema menstruacije uopšte',
        'i do not have periods at the moment',
        'ne krvarim/ nemam menstruaciju',
        'nema menstruacije uopšte (amenoreja, ne zbog trudnoće ili menopauze)'
    ]):
        return 'No menstruation'
    return 'Other'

cumulative_df['How would you describe your menstrual cycle pattern in the last year?'] = cumulative_df['How would you describe your menstrual cycle pattern in the last year?'].apply(map_cycle_pattern)
print('Cycle patterns:', cumulative_df['How would you describe your menstrual cycle pattern in the last year?'].unique())

Cycle patterns: ['Regular' 'Hormonal contraception' 'No menstruation' 'Irregular']


In [24]:
# Standardize menstrual bleeding duration
def map_bleeding_days(val):
    if pd.isnull(val):
        return None
    val = str(val).strip().lower()
    if any(x in val for x in ['1-3', '1–3']):
        return '1-3 days'
    if any(x in val for x in ['4-6', '4–6']):
        return '4-6 days'
    if any(x in val for x in ['7-8', '7–8']):
        return '7-8 days'
    if any(x in val for x in ['more than 8', 'više od  8', '8 günden']):
        return 'More than 8 days'
    if any(x in val for x in ['ne krvarim', 'tidak berdarah']):
        return 'No bleeding'
    return val

cumulative_df['How many days do you usually bleed during your period?'] = cumulative_df['How many days do you usually bleed during your period?'].apply(map_bleeding_days)
print('Bleeding days:', cumulative_df['How many days do you usually bleed during your period?'].unique())

Bleeding days: ['7-8 days' '4-6 days' '1-3 days' 'More than 8 days' 'No bleeding']


In [25]:
# Standardize symptom frequency columns (bloating, diarrhea, constipation, tiredness, etc.)
def map_symptom_frequency(val):
    if pd.isnull(val):
        return None
    val = str(val).strip().lower()
    if any(x in val for x in ['never or almost never', 'nikad ili skoro nikad', 'neredeyse hiç', 'hiçbir zaman', 'nikad ili skoro nikad ', 'neredeyse hiç ', 'hiçbir zaman ', 'nikad ili skoro nikad,', 'neredeyse hiç,', 'hiçbir zaman,']):
        return 'Never or almost never'
    if any(x in val for x in ['less than once', 'manje od jednom', 'ayda birden az', 'birden az', 'manje od jednom ', 'ayda birden az ', 'birden az ', 'less than once ', 'manje od jednom,', 'ayda birden az,', 'birden az,', 'less than once,']):
        return 'Less than once per month'
    if any(x in val for x in ['1-3 days each month', '1–3 dana', '1–3 gün', '1–3 dana svaki mjesec', '1–3 gün her ay', '1-3 dana', '1-3 gün', '1-3 dana svaki mjesec', '1-3 gün her ay', '1-3 days each month ', '1–3 dana ', '1–3 gün ', '1-3 dana ', '1-3 gün ', '1-3 days each month,', '1–3 dana,', '1–3 gün,', '1-3 dana,', '1-3 gün,']):
        return '1-3 days each month'
    if any(x in val for x in ['4-10 days', '4–10 dana', '4–10 gün', '4–10 dana svaki mjesec', '4–10 gün her ay', '4-10 dana', '4-10 gün', '4-10 dana svaki mjesec', '4-10 gün her ay', '4-10 days each month', '4-10 days each month ', '4–10 dana ', '4–10 gün ', '4-10 dana ', '4-10 gün ', '4-10 days each month,', '4–10 dana,', '4–10 gün,', '4-10 dana,', '4-10 gün,']):
        return '4-10 days each month'
    if any(x in val for x in ['more than 10', 'više od 10', 'više od  10 dana svaki mjesec', 'vişe od 10', '10 günden fazla', 'više od 10 dana', 'više od  10 dana', 'more than 10 days each month', 'more than 10 days each month ', 'više od 10 dana svaki mjesec ', 'vişe od 10 ', '10 günden fazla ', 'više od 10 dana ', 'više od  10 dana ', 'more than 10 days each month,', 'više od 10 dana svaki mjesec,', 'vişe od 10,', '10 günden fazla,', 'više od 10 dana,', 'više od  10 dana,']):
        return 'More than 10 days each month'
    # If not matched, return 'Other' to ensure English only
    return 'Other'

frequency_cols = [
    'How often do you feel very tired or exhausted, not improved by sleep? ',
    'How often do you have noticeable bloating or "endo belly"?',
    'Around your period (from 3 days before bleeding until the last day of bleeding), how often do you usually have diarrhea?',
    'Around your period (from 3 days before bleeding until the last day of bleeding), how often do you usually have constipation or very hard stools?'
]

for col in frequency_cols:
    if col in cumulative_df.columns:
        cumulative_df[col] = cumulative_df[col].apply(map_symptom_frequency)
        print(f"Unique values in '{col}':", cumulative_df[col].unique())
    else:
        print(f"Column not found: {col}")
print('Symptom frequency columns standardized')

Unique values in 'How often do you feel very tired or exhausted, not improved by sleep? ': ['More than 10 days each month' 'Other' 'Never or almost never'
 'Less than once per month' '1-3 days each month' '4-10 days each month'
 None]
Column not found: How often do you have noticeable bloating or "endo belly"?
Unique values in 'Around your period (from 3 days before bleeding until the last day of bleeding), how often do you usually have diarrhea?': ['Never or almost never' 'Other' 'Less than once per month'
 'More than 10 days each month' None '1-3 days each month'
 '4-10 days each month']
Unique values in 'Around your period (from 3 days before bleeding until the last day of bleeding), how often do you usually have constipation or very hard stools?': ['Other' 'Never or almost never' 'Less than once per month'
 'More than 10 days each month' None '4-10 days each month'
 '1-3 days each month']
Symptom frequency columns standardized


In [26]:
# Standardize pregnancy and conception related columns
def map_pregnancy_simple(val):
    if pd.isnull(val):
        return None
    val = str(val).strip().lower()
    if any(x in val for x in ['yes', 'da', 'evet', 'pregnant']):
        return 'Yes'
    if any(x in val for x in ['no', 'ne', 'hayır']):
        return 'No'
    if any(x in val for x in ['not sure', 'nisam sigurna', 'emin değilim']):
        return 'Not sure'
    return val

cumulative_df['Are you currently pregnant?'] = cumulative_df['Are you currently pregnant?'].apply(map_pregnancy_simple)

# For trying to conceive (more options)
def map_trying_conceive(val):
    if pd.isnull(val):
        return None
    val = str(val).strip().lower()
    if any(x in val for x in ['already pregnant', 'već sam trudna', 'zaten hamileyim']):
        return 'Already pregnant'
    if any(x in val for x in ['yes', 'da', 'evet']):
        return 'Yes'
    if any(x in val for x in ['no', 'ne', 'hayır']):
        return 'No'
    if any(x in val for x in ['not sure', 'nisam sigurna', 'emin değilim']):
        return 'Not sure'
    return val

cumulative_df['Are you currently trying to conceive (become pregnant)?'] = cumulative_df['Are you currently trying to conceive (become pregnant)?'].apply(map_trying_conceive)

print('Pregnancy columns standardized')

Pregnancy columns standardized


In [27]:
# Standardize bleeding/spotting related yes/no columns
def map_cycle_bleeding_yesno(val):
    if pd.isnull(val):
        return None
    val = str(val).strip().lower()
    if any(x in val for x in ['yes, occasionally', 'yes, most cycles', 'yes, more than', 'da, povremeno', 'da, više']):
        if 'most' in val or 'više' in val or 'skoro' in val:
            return 'Yes, most cycles'
        return 'Yes, occasionally'
    if any(x in val for x in ['yes', 'da', 'evet']):
        return 'Yes'
    if any(x in val for x in ['no', 'ne', 'hayır', 'nikad']):
        return 'No'
    if any(x in val for x in ['i do not have periods', 'ne krvarim', 'diye', 'adet dönemi değil']):
        return 'No periods'
    return val

bleeding_columns = [
    'In the last 12 months, have you had any bleeding or spotting between periods (not counting your normal period)?',
    'In the last 12 months, have you had long episodes of almost constant bleeding or spotting (lasting more than 3 weeks)?'
]

for col in bleeding_columns:
    if col in cumulative_df.columns:
        cumulative_df[col] = cumulative_df[col].apply(map_cycle_bleeding_yesno)

# Standardize missed periods column
def map_missed_periods(val):
    if pd.isnull(val):
        return None
    val = str(val).strip().lower()
    if any(x in val for x in ['yes, once', 'da, jednom', 'evet, bir kez']):
        return 'Yes, once'
    if any(x in val for x in ['yes, 2-3', 'da, 2-3', 'evet, 2-3']):
        return 'Yes, 2-3 times'
    if any(x in val for x in ['yes, more', 'da, više', 'evet, birden fazla']):
        return 'Yes, more than 3 times'
    if any(x in val for x in ['no', 'ne', 'hayır']):
        return 'No'
    if any(x in val for x in ['i do not remember', 'ne sjećam se', 'hatirlamiyorum']):
        return 'Do not remember'
    return val

cumulative_df['In the last 12 months, have you had cycles where your period was missed (no bleeding for more than 3 months, without pregnancy)?'] = cumulative_df['In the last 12 months, have you had cycles where your period was missed (no bleeding for more than 3 months, without pregnancy)?'].apply(map_missed_periods)

print('Bleeding/spotting columns standardized')

Bleeding/spotting columns standardized


In [28]:
# Standardize pain interference with work/activities
def map_pain_interference(val):
    if pd.isnull(val):
        return None
    val = str(val).strip().lower()
    if any(x in val for x in ['rarely', 'rijetko', 'nadiren']):
        return 'Rarely'
    if any(x in val for x in ['never', 'nikad', 'hiçbir zaman']):
        return 'Never'
    if any(x in val for x in ['some days each month', 'nekim danima', 'ayda bazı günler']):
        return 'Some days each month'
    if any(x in val for x in ['many days each month', 'većinu dana', 'ayda çoğu gün']):
        return 'Many days each month'
    if any(x in val for x in ['almost every day', 'skoro svaki dan', 'neredeyse her gün']):
        return 'Almost every day'
    return val

cumulative_df['In the last 3 months, how often did pain interfere with work, study, or daily activities?'] = cumulative_df['In the last 3 months, how often did pain interfere with work, study, or daily activities?'].apply(map_pain_interference)

# Standardize mood/anxiety/depression column
def map_mood_cycles(val):
    if pd.isnull(val):
        return None
    val = str(val).strip().lower()
    if any(x in val for x in ['never', 'nikad', 'hiçbir zaman']):
        return 'Never'
    if any(x in val for x in ['rarely', 'rijetko', 'nadiren']):
        return 'Rarely'
    if any(x in val for x in ['some cycles', 'poneki ciklus', 'çoğu döngüde', 'bazı']):
        return 'Some cycles'
    if any(x in val for x in ['most cycles', 'većinu ciklusa', 'çoğu döngüs', 'her döngüdü']):
        return 'Most cycles'
    if any(x in val for x in ['every cycle', 'svaki ciklus', 'her döngücü']):
        return 'Every cycle'
    return val

cumulative_df['In the last 6 months, how often have you had mood swings, anxiety or depression clearly related to your cycle? '] = cumulative_df['In the last 6 months, how often have you had mood swings, anxiety or depression clearly related to your cycle? '].apply(map_mood_cycles)

print('Pain interference and mood columns standardized')

Pain interference and mood columns standardized


In [29]:
# Standardize symptom onset timeline
def map_symptom_onset(val):
    if pd.isnull(val):
        return None
    val = str(val).strip().lower()
    if any(x in val for x in ['less than', 'manje od godinu', 'ayda 1 yıldan']):
        return 'Less than 1 year ago'
    if any(x in val for x in ['1-3 years', '1-3 godine', '1–3 yıl önce']):
        return '1-3 years ago'
    if any(x in val for x in ['3-7 years', 'prije 3-7 godina', '3–7 yıl']):
        return '3-7 years ago'
    if any(x in val for x in ['more than 7', 'prije više od 7', '7 yıldan fazla']):
        return 'More than 7 years ago'
    if any(x in val for x in ['i don\'t have any symptoms', 'nemam nikakve', 'semptomum yok', 'simptomim nema']):
        return 'No symptoms'
    if any(x in val for x in ['i don\'t remember', 'ne sjećam se', 'hatirlamiyorum']):
        return 'Do not remember'
    return val

cumulative_df['How long ago did your symptoms (pain, cycle changes or other main problems) first start?'] = cumulative_df['How long ago did your symptoms (pain, cycle changes or other main problems) first start?'].apply(map_symptom_onset)

# Standardize time to diagnosis columns
def map_time_to_diagnosis(val):
    if pd.isnull(val):
        return None
    val = str(val).strip().lower()
    if any(x in val for x in ['i have symptoms but no one has mentioned', 'imam simptome ali niko nije', 'semptomlar var ama']):
        return 'Have symptoms, not mentioned'
    if any(x in val for x in ['less than 2 years', 'manje od 2 godine', '2 yıldan az']):
        return 'Less than 2 years'
    if any(x in val for x in ['2-5 years', '2–5 godina', '2–5 yıl']):
        return '2-5 years'
    if any(x in val for x in ['more than 5', 'više od 5', '5 yıldan fazla']):
        return 'More than 5 years'
    if any(x in val for x in ['0 years', 'iste godine']):
        return 'Same year as symptoms'
    if any(x in val for x in ['not sure', 'nisam sigurna', 'emin değilim', 'hatirlamiyorum']):
        return 'Not sure'
    return val

cumulative_df['If you have endometriosis or a doctor suspects it: How many years passed between your first symptoms and the first time a doctor mentioned or diagnosed endometriosis?'] = cumulative_df['If you have endometriosis or a doctor suspects it: How many years passed between your first symptoms and the first time a doctor mentioned or diagnosed endometriosis?'].apply(map_time_to_diagnosis)

cumulative_df['If you have PCOS or a doctor suspects it: How many years passed between your first symptoms and the first time a doctor mentioned or diagnosed PCOS?'] = cumulative_df['If you have PCOS or a doctor suspects it: How many years passed between your first symptoms and the first time a doctor mentioned or diagnosed PCOS?'].apply(map_time_to_diagnosis)

print('Symptom onset and diagnosis timeline columns standardized')

Symptom onset and diagnosis timeline columns standardized


In [30]:
# Convert pain scales and remedy effectiveness scores to float
pain_scale_cols = [
    'Period pain / cramps during your period (dysmenorrhea)',
    'Pelvic pain outside your period (chronic pelvic pain)\n',
    'Abdominal pain or pressure (not only during period)\n',
    'Lower back pain related to your cycle',
    'Hip or leg pain related to your cycle\n',
    'Sharp or stabbing pelvic/abdominal pain',
    'Painful ovulation (mid‑cycle pain)\n',
    'Overall chronic pain level in the last 3 months (all pains combined)\n',
    'Overall, how much do your symptoms affect your quality of life? (0 = Not at all, 10 = Extremely)'
]

for col in pain_scale_cols:
    if col in cumulative_df.columns:
        cumulative_df[col] = cumulative_df[col].astype(float)

# Convert remedy effectiveness scores to float
remedy_cols = [
    'Heat (heating pad, hot water bottle, warm baths)',
    'Herbal teas and medicinal herbs overall',
    'Chamomile (Kamilica) tea',
    'Ginger (Đumbir) tea\n',
    'Peppermint (Nana/Menta) tea\n',
    'Myo‑inositol',
    'Magnesium',
    'Vitamin D',
    'Omega‑3',
    'Gentle stretching / mobility',
    'Yoga or Pilates',
    'Walking',
    'Higher protein diet',
    'Anti‑inflammatory diet',
    'Reduced sugar / sweets'
]

for col in remedy_cols:
    if col in cumulative_df.columns:
        cumulative_df[col] = cumulative_df[col].astype(float)

# Update and save the dataset
cumulative_df.to_csv('../data/cumulative_data.csv', index=False)
print('All standardizations complete. Dataset saved.')

All standardizations complete. Dataset saved.


In [31]:
# Save cumulative_df to optimized_dataset.csv
cumulative_df.to_csv("../data/optimized_dataset.csv", index=False)
print("Saved cumulative_df to ../data/optimized_dataset.csv")

Saved cumulative_df to ../data/optimized_dataset.csv


In [32]:
optimized_df = pd.read_csv("../data/optimized_dataset.csv")


In [33]:
# print What is your height (in cm)? unique values
print('Unique values in height column:', optimized_df['What is your height (in cm)?'].unique())

# remove any letters and if there is . or , replace with empty and convert to numeric
optimized_df['What is your height (in cm)?'] = (
    optimized_df['What is your height (in cm)?']
    .astype(str)
    .str.replace(r'[^\d.,]', '', regex=True)
    .str.replace(',', '', regex=False)
    .replace('', None)
    .astype(float)
    .round(0)
    .astype('Int64')
)
print('Unique values in height column after cleaning:', optimized_df['What is your height (in cm)?'].unique())


# update optimized_df and save to optimized_dataset.csv
optimized_df.to_csv("../data/optimized_dataset.csv", index=False)
print("Updated optimized_df saved to ../data/optimized_dataset.csv")

Unique values in height column: ['168' '157' '158' '178' '183' '163' '167' '165' '1,61' '156' '161'
 '170cm' '159cm' '170' '166' '172 cm' '160' '172' '150' '156cm' '159'
 '171' '174' '177' '169' '175' '180' '179' '164' '155' '185' '162' '181'
 '160cm' '168cm' '173' '169cm' '1u0' '166cm' '162cm' '180cm']
Unique values in height column after cleaning: <IntegerArray>
[168, 157, 158, 178, 183, 163, 167, 165, 161, 156, 170, 159, 166, 172, 160,
 150, 171, 174, 177, 169, 175, 180, 179, 164, 155, 185, 162, 181, 173,  10]
Length: 30, dtype: Int64
Updated optimized_df saved to ../data/optimized_dataset.csv


In [34]:
# print What is your weight (in kg)? unique values
print('Unique values in weight column:', optimized_df['What is your weight (in kg)?'].unique())


Unique values in weight column: [ 55  41  53  60  50  52  97  63  56  49  80  79  58  68  69  75  77  65
  83  71  64  72  66  62  67  70  48  57  59 108  90  61  54  73  82 100
  76  81  85  51 104  78 115  47  88]


In [35]:
# Improved mapping for Bosnian and Turkish race/ethnicity values to English
def map_race_ethnicity(val):
    if pd.isnull(val):
        return None
    val = str(val).strip().lower()
    # English and Bosnian/Turkish mappings
    if (
        ('white' in val and ('slavic' in val or 'isto' in val or 'doğu' in val))
        or ('bijela' in val and ('slavenska' in val or 'istočno' in val or 'srednjoevropska' in val))
        or ('beyaz' in val and 'slav' in val)
    ):
        return 'White – Slavic / Eastern or Central European'
    if (
        ('white' in val and ('western' in val or 'zapadno' in val or 'sjeverno' in val or 'northern' in val))
        or ('bijela' in val and ('zapadno' in val or 'sjevernoevropska' in val or 'sjeverno' in val))
        or ('beyaz' in val and ('western' in val or 'zapad' in val or 'sjeverno' in val or 'northern' in val))
    ):
        return 'White – Western / Northern European'
    if (
        'middle eastern' in val or 'north african' in val or 'west asian' in val
        or 'orta doğulu' in val or 'kuzey afrikalı' in val or 'batı asyalı' in val or 'sjevernoafrička' in val
    ):
        return 'Middle Eastern / North African / West Asian'
    if 'hispanic' in val or 'latin' in val:
        return 'Hispanic / Latin American'
    if (
        'black' in val or 'crna' in val or 'afrička' in val or 'african' in val
    ):
        return 'Black / African / African descent'
    if (
        'mixed' in val or 'miješana' in val or 'multiple' in val
    ):
        return 'Mixed / multiple backgrounds'
    if (
        'other' in val or 'ostalo' in val
    ):
        return 'Other (please specify)'
    if (
        'radije ne želim odgovoriti' in val or 'belirtmek istemiyorum' in val or 'prefer not to say' in val
    ):
        return None
    if val in ["", "nan"]:
        return None
    # Handle specific multi-value Bosnian/Turkish cases
    if val == 'bijela – slavenska / istočno- ili srednjoevropska, bijela – zapadno- ili sjevernoevropska':
        return 'White – Slavic / Eastern or Central European; White – Western / Northern European'
    if val == 'bijela – slavenska / istočno- ili srednjoevropska':
        return 'White – Slavic / Eastern or Central European'
    if val == 'bijela – zapadno- ili sjevernoevropska, južnoazijska':
        return 'White – Western / Northern European; South Asian'
    if val == 'bijela – zapadno- ili sjevernoevropska':
        return 'White – Western / Northern European'
    if val == 'beyaz – slav / doğu veya orta avrupa':
        return 'White – Slavic / Eastern or Central European'
    # If not matched, return the original value (preserve data)
    return val

optimized_df['Which race or ethnic group(s) do you identify with? (Optional, select all that apply)'] = optimized_df['Which race or ethnic group(s) do you identify with? (Optional, select all that apply)'].apply(map_race_ethnicity)

print('Unique values in race/ethnicity column:', optimized_df['Which race or ethnic group(s) do you identify with? (Optional, select all that apply)'].unique())

# update optimized_df and save to optimized_dataset.csv
optimized_df.to_csv("../data/optimized_dataset.csv", index=False)
print("Updated optimized_df saved to ../data/optimized_dataset.csv")


Unique values in race/ethnicity column: ['White – Slavic / Eastern or Central European'
 'Middle Eastern / North African / West Asian'
 'White – Western / Northern European' 'Mixed / multiple backgrounds'
 'Other (please specify)' 'Black / African / African descent' None]
Updated optimized_df saved to ../data/optimized_dataset.csv


In [36]:
# Standardize 'first period age' column to numeric age, with explicit grade mapping (e.g., '7 razred' = 12, etc.)
import re
def extract_first_period_age(val):
    if pd.isnull(val):
        return None
    val = str(val).strip().lower()
    # Handle explicit grade/razred cases
    if '5 razred' in val or 'peti razred' in val or '5. razred' in val or 'petog razreda' in val:
        return 11
    if '6 razred' in val or 'šesti razred' in val or '6. razred' in val or 'sesti razred' in val:
        return 12
    if '7 razred' in val or 'sedmi razred' in val or '7. razred' in val:
        return 13
    if '8 razred' in val or 'osmi razred' in val or '8. razred' in val:
        return 14
    if '9 razred' in val or 'deveti razred' in val or '9. razred' in val:
        return 15
    # Handle common grade/razred/razreda/razredom/razredu cases
    if 'razred' in val:
        # fallback: just extract any number, but grades above handled above
        pass
    # Extract all numbers (including decimals)
    nums = re.findall(r'\d+[\.,]?\d*', val)
    if nums:
        # If range, take the lower end
        if '-' in val or 'do' in val or 'ili' in val:
            try:
                return float(nums[0].replace(',', '.'))
            except:
                return None
        # If 'sa' or 'u' (at age), use the number
        if 'sa' in val or 'u' in val:
            try:
                return float(nums[0].replace(',', '.'))
            except:
                return None
        # If 'godina' or 'years' or 'yrs', use the number
        if 'godina' in val or 'years' in val or 'yrs' in val:
            try:
                return float(nums[0].replace(',', '.'))
            except:
                return None
        # If only one number, use it
        try:
            return float(nums[0].replace(',', '.'))
        except:
            return None
    # Fallback for known phrases
    if 'nisam sigurna' in val:
        return None
    return None

optimized_df['At what age did you have your first period?'] = optimized_df['At what age did you have your first period?'].apply(extract_first_period_age)
print('Unique values in first period age column after cleaning:', optimized_df['At what age did you have your first period?'].unique())

# Save changes
optimized_df.to_csv("../data/optimized_dataset.csv", index=False)
print("Updated optimized_df saved to ../data/optimized_dataset.csv")


Unique values in first period age column after cleaning: [13.  12.  14.  11.  10.5 15.   9.  10.   nan 26.  11.5 12.5 16. ]
Updated optimized_df saved to ../data/optimized_dataset.csv


In [37]:
# Standardize menstrual cycle length for optimized_df

def map_cycle_length_optimized(val):
    if pd.isnull(val):
        return None
    val = str(val).strip().lower()
    if any(x in val for x in [
        'less than 21',
        'manje od 21',
        'manje od 21 dana',
        'ayda 21 günden az',
        '21 günden az'
    ]):
        return 'Less than 21 days'
    if any(x in val for x in [
        '21–27',
        '21-27',
        '21–27 dana',
        '21-27 dana',
        '21–27 gün',
        '21-27 gün'
    ]):
        return '21-27 days'
    if any(x in val for x in [
        '28-34',
        '28–34',
        '28-34 dana',
        '28–34 dana',
        '28–34 gün',
        '28-34 gün'
    ]):
        return '28-34 days'
    if any(x in val for x in [
        '35-60',
        '35–60',
        '35-60 dana',
        '35–60 dana',
        '35–60 gün',
        '35-60 gün'
    ]):
        return '35-60 days'
    if any(x in val for x in [
        'more than 60',
        'više od 60',
        '60 günden fazla',
        'više od 60 dana',
        '60 günden fazla'
    ]):
        return 'More than 60 days'
    if any(x in val for x in [
        'not sure',
        'nisam sigurna',
        'emin değilim',
        'nisam sigurna',
        'ne znam',
        'hatırlamıyorum'
    ]):
        return 'Not sure'
    return 'Other'

optimized_df['How long is your typical menstrual cycle? (Number of days from Day 1 to the next Day 1)'] = optimized_df['How long is your typical menstrual cycle? (Number of days from Day 1 to the next Day 1)'].apply(map_cycle_length_optimized)
print('Cycle lengths (optimized):', optimized_df['How long is your typical menstrual cycle? (Number of days from Day 1 to the next Day 1)'].unique())


# Save changes
optimized_df.to_csv("../data/optimized_dataset.csv", index=False)
print("Updated optimized_df saved to ../data/optimized_dataset.csv")

Cycle lengths (optimized): ['28-34 days' '21-27 days' '35-60 days' 'Not sure' 'Less than 21 days'
 'More than 60 days']
Updated optimized_df saved to ../data/optimized_dataset.csv


In [38]:
# print unique values in Do you experience heavy or extreme menstrual bleeding (for example needing to change pads/tampons every 1–2 hours on the heaviest day, or using double protection)?',

print('Unique values in heavy bleeding column:', optimized_df['Do you experience heavy or extreme menstrual bleeding (for example needing to change pads/tampons every 1–2 hours on the heaviest day, or using double protection)?'].unique())  

Unique values in heavy bleeding column: ['No' 'Sometimes' 'Almost every period' 'Often' 'Ne' 'Ponekad' 'Često'
 'Skoro svaki ciklus' 'Bazen']


In [39]:
# Map heavy bleeding responses to English and update the column in place (using the exact column name)

def map_heavy_bleeding(val):
    if pd.isnull(val):
        return None
    val = str(val).strip().lower()
    if val in ['no', 'ne']:
        return 'No'
    if val in ['sometimes', 'ponekad', 'bazen']:
        return 'Sometimes'
    if val in ['almost every period', 'skoro svaki ciklus']:
        return 'Almost every period'
    if val in ['often', 'često', 'cesto']:
        return 'Often'
    return val.capitalize()

col_name = 'Do you experience heavy or extreme menstrual bleeding (for example needing to change pads/tampons every 1–2 hours on the heaviest day, or using double protection)?'
optimized_df[col_name] = optimized_df[col_name].apply(map_heavy_bleeding)
print('Unique values in heavy bleeding column (EN):', optimized_df[col_name].unique())

# Save changes
optimized_df.to_csv("../data/optimized_dataset.csv", index=False)
print("Updated optimized_df saved to ../data/optimized_dataset.csv")


Unique values in heavy bleeding column (EN): ['No' 'Sometimes' 'Almost every period' 'Often']
Updated optimized_df saved to ../data/optimized_dataset.csv


In [40]:
# print unique values in In the last 12 months, have you had cycles where your period was missed (no bleeding for more than 3 months, without pregnancy)?',

print('Unique values in missed periods column:', optimized_df['In the last 12 months, have you had cycles where your period was missed (no bleeding for more than 3 months, without pregnancy)?'].unique()) 

# adapt to english

Unique values in missed periods column: ['No' 'Yes, more than 3 times' 'Yes, once' 'Yes, 2-3 times'
 'evet, 2–3 kez' 'hatırlamıyorum']


In [41]:
# Standardize missed periods responses to English for the exact survey column
def map_missed_periods_standardized(val):
    if pd.isnull(val):
        return None
    val = str(val).strip().lower()

    if val in ['no', 'ne', 'hayır', 'hayir']:
        return 'No'
    if val in [
        'yes, more than 3 times',
        'da, više od 3 puta',
        'evet, 3 kereden fazla',
        'yes, more than three times'
    ]:
        return 'Yes, more than 3 times'
    if val in ['yes, once', 'da, jednom', 'evet, bir kez']:
        return 'Yes, once'
    if val in [
        'yes, 2-3 times',
        'yes, 2–3 times',
        'da, 2-3 puta',
        'da, 2–3 puta',
        'evet, 2-3 kez',
        'evet, 2–3 kez'
    ]:
        return 'Yes, 2-3 times'
    if val in [
        "i don't remember",
        "don't remember",
        'do not remember',
        'ne sjećam se',
        'ne sjecam se',
        'hatırlamıyorum',
        'hatirlamiyorum',
        'nisam sigurna',
        'not sure'
    ]:
        return "Don't remember"

    return 'Other'

col_missed = 'In the last 12 months, have you had cycles where your period was missed (no bleeding for more than 3 months, without pregnancy)?'
optimized_df[col_missed] = optimized_df[col_missed].apply(map_missed_periods_standardized)
print('Unique values in missed periods column (EN):', optimized_df[col_missed].dropna().unique())

# Save changes
optimized_df.to_csv("../data/optimized_dataset.csv", index=False)
print("Updated optimized_df saved to ../data/optimized_dataset.csv")

Unique values in missed periods column (EN): ['No' 'Yes, more than 3 times' 'Yes, once' 'Yes, 2-3 times'
 "Don't remember"]
Updated optimized_df saved to ../data/optimized_dataset.csv


In [42]:
# Standardize bloating/"endo belly" column in optimized_df to English categories
def map_bloating_frequency(val):
    if pd.isnull(val):
        return None
    val = str(val).strip().lower()

    if any(x in val for x in [
        'never or almost never',
        'nikad ili skoro nikad',
        'hiçbir zaman',
        'neredeyse hiç'
    ]):
        return 'Never or almost never'
    if any(x in val for x in [
        'less than once per month',
        'less than once',
        'manje od jednom mjesečno',
        'manje od jednom',
        'ayda birden az'
    ]):
        return 'Less than once per month'
    if any(x in val for x in [
        '1-3 days each month',
        '1–3 days each month',
        '1-3 dana svaki mjesec',
        '1–3 dana svaki mjesec',
        'ayda 1-3 gün',
        'ayda 1–3 gün'
    ]):
        return '1-3 days each month'
    if any(x in val for x in [
        '4-10 days each month',
        '4–10 days each month',
        '4-10 dana svaki mjesec',
        '4–10 dana svaki mjesec',
        'ayda 4-10 gün',
        'ayda 4–10 gün'
    ]):
        return '4-10 days each month'
    if any(x in val for x in [
        'more than 10 days each month',
        'more than 10',
        'više od 10 dana svaki mjesec',
        'više od 10',
        'ayda 10 günden fazla',
        '10 günden fazla'
    ]):
        return 'More than 10 days each month'
    if any(x in val for x in [
        "don't remember",
        'do not remember',
        'ne sjećam se',
        'ne sjecam se',
        'hatırlamıyorum',
        'hatirlamiyorum',
        'not sure',
        'nisam sigurna',
        'emin değilim'
    ]):
        return "Don't remember"

    return 'Other'

bloating_candidates = [
    'How often do you have noticeable bloating or "endo belly"?',
    'How often do you have noticeable bloating or “endo belly”?'
]

bloating_col = next((c for c in bloating_candidates if c in optimized_df.columns), None)
if bloating_col is None:
    print('Bloating column not found in optimized_df')
else:
    optimized_df[bloating_col] = optimized_df[bloating_col].apply(map_bloating_frequency)
    print('Unique values in bloating column (EN):', optimized_df[bloating_col].dropna().unique())
    optimized_df.to_csv('../data/optimized_dataset.csv', index=False)
    print('Updated optimized_df saved to ../data/optimized_dataset.csv')

Unique values in bloating column (EN): ['1-3 days each month' 'Less than once per month'
 'More than 10 days each month' '4-10 days each month'
 'Never or almost never' 'Other']
Updated optimized_df saved to ../data/optimized_dataset.csv


In [43]:
# Standardize multi-select diagnosis-history column to English labels (preserve free-text Other values)
import re

def _other_label_from_token(token):
    raw = str(token).strip()
    cleaned = raw
    while re.match(r'^(other|ostalo|drugo|diğer|diger)\s*[:\-]?\s*', cleaned, flags=re.IGNORECASE):
        cleaned = re.sub(r'^(other|ostalo|drugo|diğer|diger)\s*[:\-]?\s*', '', cleaned, flags=re.IGNORECASE).strip()
    return f'Other: {cleaned}' if cleaned else 'Other'

def standardize_told_conditions(value):
    if pd.isnull(value):
        return None

    text = str(value).strip().lower()
    if text in ['', 'nan', 'none']:
        return None

    normalized = text.replace('\n', ',').replace(';', ',').replace('|', ',').replace('/', ',')
    parts = [p.strip() for p in normalized.split(',') if p.strip()]
    if not parts:
        parts = [normalized.strip()]

    labels = set()

    for p in parts:
        p_clean = re.sub(r'\s+', ' ', p).strip()

        if any(x in p_clean for x in ['endometriosis', 'endometrioza', 'endometriozis']):
            labels.add('Endometriosis')
            continue
        if any(x in p_clean for x in ['pcos', 'polycystic ovary', 'polikistic', 'polikistik']):
            labels.add('PCOS')
            continue
        if any(x in p_clean for x in ['adenomyosis', 'adenomioza', 'adenomyozis']):
            labels.add('Adenomyosis')
            continue
        if any(x in p_clean for x in ['fibroid', 'myoma', 'miom', 'miyom']):
            labels.add('Fibroids')
            continue
        if any(x in p_clean for x in ['irritable bowel', 'ibs', 'sindrom iritabilnog', 'hassas bagirsak']):
            labels.add('IBS')
            continue
        if any(x in p_clean for x in ['thyroid', 'štitnja', 'stitnja', 'tiroid']):
            labels.add('Thyroid disorder')
            continue
        if any(x in p_clean for x in ['insulin resistance', 'inzulinska rezistencija', 'insülin direnci', 'insulin direnci']):
            labels.add('Insulin resistance')
            continue
        if any(x in p_clean for x in ['infertility', 'neplod', 'kısırlık', 'kisirlik', 'subfertility']):
            labels.add('Infertility/subfertility')
            continue

        if any(x in p_clean for x in ['none', 'no condition', 'ne', 'nemam', 'hayır', 'hayir', 'nista', 'ništa', 'nothing', 'not diagnosed']):
            labels.add('None')
            continue
        if any(x in p_clean for x in ["don't know", 'do not know', 'not sure', 'nisam sigurna', 'ne znam', 'emin değilim', 'hatırlamıyorum', 'hatirlamiyorum']):
            labels.add("Don't know")
            continue
        if any(x in p_clean for x in ['other', 'ostalo', 'drugo', 'diğer', 'diger']):
            labels.add(_other_label_from_token(p_clean))
            continue

        labels.add(_other_label_from_token(p_clean))

    condition_labels = {
        'Endometriosis', 'PCOS', 'Adenomyosis', 'Fibroids',
        'IBS', 'Thyroid disorder', 'Insulin resistance', 'Infertility/subfertility'
    }
    if labels.intersection(condition_labels):
        labels.discard('None')

    ordered = [
        'Endometriosis', 'PCOS', 'Adenomyosis', 'Fibroids',
        'IBS', 'Thyroid disorder', 'Insulin resistance', 'Infertility/subfertility',
        'None', "Don't know"
    ]
    result = [label for label in ordered if label in labels]
    other_values = sorted([x for x in labels if x.startswith('Other: ')])
    if 'Other' in labels and not other_values:
        result.append('Other')
    result.extend(other_values)
    return '; '.join(result) if result else None

target_col = 'Have you been told you have any of the following? (select all that apply)'
if target_col in optimized_df.columns:
    optimized_df[target_col] = optimized_df[target_col].apply(standardize_told_conditions)
    print('Unique standardized values:', optimized_df[target_col].dropna().unique())
    optimized_df.to_csv('../data/optimized_dataset.csv', index=False)
    print('Updated optimized_df saved to ../data/optimized_dataset.csv')
else:
    print(f'Column not found: {target_col}')

Unique standardized values: ['None'
 'Endometriosis; IBS; Thyroid disorder; Other: chronic pain condition'
 'Endometriosis' 'PCOS; Other: ovarian cysts'
 'Endometriosis; Other: ovarian cysts'
 'IBS; Thyroid disorder; Other: inflammatory bowel disease (ibd); Other: ovarian cysts'
 'PCOS' 'Endometriosis; Adenomyosis' 'PCOS; Thyroid disorder'
 'Thyroid disorder; Other: chronic pain condition'
 'Endometriosis; PCOS; Adenomyosis; Other: ovarian cysts; Other: pelvic inflammatory disease'
 'Endometriosis; Thyroid disorder; Other: chronic pain condition'
 'PCOS; Other: chocolate cyst; Other: endometrioma; Other: ovarian cysts'
 'Endometriosis; Fibroids' 'IBS; Thyroid disorder' 'Other: ovarian cysts'
 'Other: chronic pain condition'
 'Other: ciste na jajnicima; Other: endomentrioza' 'Thyroid disorder'
 'Fibroids; IBS; Thyroid disorder; Other: autoimuna bolest (npr. lupus; Other: ciste na jajnicima; Other: reumatoidni artritis)'
 'Adenomyosis; Other: endomentrioza'
 'Fibroids; Thyroid disorder; 

In [44]:
# Standardize hormone-related features and pelvic ultrasound findings columns (preserve free-text Other values)
import re

def _split_multiselect(value):
    if pd.isnull(value):
        return []
    text = str(value).strip()
    if text.lower() in ['', 'nan', 'none']:
        return []
    normalized = (
        text.replace('\n', ',')
            .replace(';', ',')
            .replace('|', ',')
            .replace('/', ',')
    )
    parts = [p.strip() for p in normalized.split(',') if p.strip()]
    return parts if parts else [normalized.strip()]

def _resolve_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    def norm(s):
        return (
            s.lower()
             .replace('’', "'")
             .replace('“', '"')
             .replace('”', '"')
             .replace('‑', '-')
             .replace('–', '-')
             .replace('—', '-')
             .replace('  ', ' ')
.strip()
        )
    cand_norm = {norm(c): c for c in candidates}
    for col in df.columns:
        if norm(col) in cand_norm:
            return col
    return None

def _other_label_from_token(token):
    raw = str(token).strip()
    cleaned = raw
    while re.match(r'^(other|ostalo|drugo|diğer|diger)\s*[:\-]?\s*', cleaned, flags=re.IGNORECASE):
        cleaned = re.sub(r'^(other|ostalo|drugo|diğer|diger)\s*[:\-]?\s*', '', cleaned, flags=re.IGNORECASE).strip()
    return f'Other: {cleaned}' if cleaned else 'Other'

def standardize_hormone_features(value):
    parts = _split_multiselect(value)
    if not parts:
        return None

    labels = set()
    for p in parts:
        pl = p.lower()
        if any(x in pl for x in ['acne', 'akne']):
            labels.add('Acne/oily skin')
            continue
        if any(x in pl for x in ['facial hair', 'body hair', 'hirsut', 'pojačana dlakavost', 'dlakavost', 'aşırı tüylenme', 'asiri tuylenme']):
            labels.add('Excess facial/body hair')
            continue
        if any(x in pl for x in ['hair thinning', 'hair loss', 'alopecia', 'opadanje kose', 'proređivanje kose', 'sac dokulmesi', 'saç dökülmesi']):
            labels.add('Hair thinning/hair loss')
            continue
        if any(x in pl for x in ['weight gain', 'teško gubim težinu', 'tesko gubim tezinu', 'kilo alma', 'kilo artışı', 'kilo artisi']):
            labels.add('Weight gain/difficulty losing weight')
            continue
        if any(x in pl for x in ['dark skin', 'dark patches', 'acanthosis', 'tamne mrlje', 'kararma', 'koyu lekeler']):
            labels.add('Dark skin patches (acanthosis)')
            continue
        if any(x in pl for x in ['skin tag', 'fibroepithelial', 'viseći mladeži', 'viseci mladezi', 'et beni']):
            labels.add('Skin tags')
            continue
        if any(x in pl for x in ['none', 'no', 'nemam', 'ne', 'hayır', 'hayir', 'nothing']):
            labels.add('None')
            continue
        if any(x in pl for x in ["don't know", 'do not know', 'not sure', 'nisam sigurna', 'ne znam', 'emin değilim', 'hatırlamıyorum', 'hatirlamiyorum']):
            labels.add("Don't know")
            continue
        if any(x in pl for x in ['other', 'ostalo', 'drugo', 'diğer', 'diger']):
            labels.add(_other_label_from_token(p))
            continue
        labels.add(_other_label_from_token(p))

    condition_labels = {'Acne/oily skin', 'Excess facial/body hair', 'Hair thinning/hair loss', 'Weight gain/difficulty losing weight', 'Dark skin patches (acanthosis)', 'Skin tags'}
    if labels.intersection(condition_labels):
        labels.discard('None')

    ordered = ['Acne/oily skin', 'Excess facial/body hair', 'Hair thinning/hair loss', 'Weight gain/difficulty losing weight', 'Dark skin patches (acanthosis)', 'Skin tags', 'None', "Don't know"]
    result = [x for x in ordered if x in labels]
    other_values = sorted([x for x in labels if x.startswith('Other: ')])
    if 'Other' in labels and not other_values:
        result.append('Other')
    result.extend(other_values)
    return '; '.join(result) if result else None

def standardize_ultrasound_findings(value):
    parts = _split_multiselect(value)
    if not parts:
        return None

    labels = set()
    for p in parts:
        pl = p.lower()
        if any(x in pl for x in ['polycystic ovary', 'pcos ovar', 'polikistič', 'polikistic', 'çoklu kist', 'coklu kist']):
            labels.add('Polycystic ovaries')
            continue
        if any(x in pl for x in ['endometrioma', 'chocolate cyst', 'čokoladna cista', 'cokoladna cista', 'cikolata kisti']):
            labels.add('Endometrioma/ovarian cyst')
            continue
        if any(x in pl for x in ['ovarian cyst', 'cista na jajniku', 'jajnička cista', 'jajnicka cista', 'yumurtalik kisti']):
            labels.add('Ovarian cyst')
            continue
        if any(x in pl for x in ['fibroid', 'myoma', 'miom', 'miyom']):
            labels.add('Fibroids')
            continue
        if any(x in pl for x in ['adenomyosis', 'adenomioza', 'adenomyozis']):
            labels.add('Adenomyosis')
            continue
        if any(x in pl for x in ['normal', 'no findings', 'without findings', 'uredan', 'normalan nalaz', 'normal bulgu yok']):
            labels.add('No abnormal findings')
            continue
        if any(x in pl for x in ['none', 'nemam', 'hayır', 'hayir', 'nothing']):
            labels.add('None')
            continue
        if any(x in pl for x in ["don't know", 'do not know', 'not sure', 'nisam sigurna', 'ne znam', 'emin değilim', 'hatırlamıyorum', 'hatirlamiyorum']):
            labels.add("Don't know")
            continue
        if any(x in pl for x in ['other', 'ostalo', 'drugo', 'diğer', 'diger']):
            labels.add(_other_label_from_token(p))
            continue
        labels.add(_other_label_from_token(p))

    finding_labels = {'Polycystic ovaries', 'Endometrioma/ovarian cyst', 'Ovarian cyst', 'Fibroids', 'Adenomyosis', 'No abnormal findings'}
    if labels.intersection(finding_labels):
        labels.discard('None')

    ordered = ['Polycystic ovaries', 'Endometrioma/ovarian cyst', 'Ovarian cyst', 'Fibroids', 'Adenomyosis', 'No abnormal findings', 'None', "Don't know"]
    result = [x for x in ordered if x in labels]
    other_values = sorted([x for x in labels if x.startswith('Other: ')])
    if 'Other' in labels and not other_values:
        result.append('Other')
    result.extend(other_values)
    return '; '.join(result) if result else None

hormone_candidates = [
    'Do you have any of these hormone-related features? (select all that apply)',
    'Do you have any of these hormone‑related features? (select all that apply)',
    'Do you have any of these hormone related features? (select all that apply)'
]
ultrasound_candidates = [
    'Have you ever had a pelvic ultrasound? If yes, were you ever told any of these findings? (select all that apply)',
    'Have you ever had a pelvic ultrasound? If yes, were you ever told any of these findings? (Select all that apply)'
]

hormone_col = _resolve_column(optimized_df, hormone_candidates)
ultrasound_col = _resolve_column(optimized_df, ultrasound_candidates)

if hormone_col is None:
    print('Hormone-features column not found in optimized_df')
else:
    optimized_df[hormone_col] = optimized_df[hormone_col].apply(standardize_hormone_features)
    print('Unique standardized hormone-feature values:', optimized_df[hormone_col].dropna().unique())

if ultrasound_col is None:
    print('Pelvic-ultrasound findings column not found in optimized_df')
else:
    optimized_df[ultrasound_col] = optimized_df[ultrasound_col].apply(standardize_ultrasound_findings)
    print('Unique standardized ultrasound-finding values:', optimized_df[ultrasound_col].dropna().unique())

if hormone_col is not None or ultrasound_col is not None:
    optimized_df.to_csv('../data/optimized_dataset.csv', index=False)
    print('Updated optimized_df saved to ../data/optimized_dataset.csv')

Unique standardized hormone-feature values: ['None'
 'Acne/oily skin; Hair thinning/hair loss; Other: armpits; Other: folds)'
 'Acne/oily skin; Other: Excess hair growth (chin; Other: abdomen; Other: chest; Other: thighs); Other: upper lip'
 'Acne/oily skin'
 'None; Other: Excess hair growth (chin; Other: abdomen; Other: armpits; Other: chest; Other: folds); Other: thighs); Other: upper lip'
 'Hair thinning/hair loss'
 'Weight gain/difficulty losing weight; Other: Difficulty losing weight'
 'Weight gain/difficulty losing weight; Other: Difficulty losing weight; Other: Excess hair growth (chin; Other: abdomen; Other: chest; Other: thighs); Other: upper lip'
 'Other: Difficulty losing weight'
 'Hair thinning/hair loss; Weight gain/difficulty losing weight; Other: Difficulty losing weight; Other: Excess hair growth (chin; Other: abdomen; Other: chest; Other: thighs); Other: upper lip'
 'Acne/oily skin; Other: Difficulty losing weight'
 'Hair thinning/hair loss; Weight gain/difficulty losi

In [45]:
# Standardize: How long ago did your symptoms first start? (preserve free-text Other values)
col_symptom_onset = 'How long ago did your symptoms (pain, cycle changes or other main problems) first start?'

if col_symptom_onset in optimized_df.columns:
    def map_symptom_onset_optimized(val):
        if pd.isnull(val):
            return None
        original = str(val).strip()
        source = re.sub(r'^(other\s*:\s*)+', '', original, flags=re.IGNORECASE).strip()
        v = source.lower()
        if any(x in v for x in ['less than 1 year', 'less than one year', 'manje od godinu', '1 yıldan az', '1 yildan az']):
            return 'Less than 1 year ago'
        if any(x in v for x in ['1-3 years', '1–3 years', '1-3 godine', '1–3 godine', '1-3 yıl', '1–3 yıl', '1-3 yil', '1–3 yil']):
            return '1-3 years ago'
        if any(x in v for x in ['3-7 years', '3–7 years', '3-7 godine', '3–7 godine', '3-7 yıl', '3–7 yıl', '3-7 yil', '3–7 yil']):
            return '3-7 years ago'
        if any(x in v for x in ['more than 7', 'više od 7', 'vise od 7', '7 yıldan fazla', '7 yildan fazla']):
            return 'More than 7 years ago'
        if any(x in v for x in ["i don't have any symptoms", 'nemam nikakve simptome', 'simptomum yok', 'no symptoms']):
            return 'No symptoms'
        if any(x in v for x in ["i don't remember", "don't remember", 'not sure', 'ne sjećam se', 'ne sjecam se', 'nisam sigurna', 'hatırlamıyorum', 'hatirlamiyorum', 'emin değilim']):
            return "Don't remember"
        return f'Other: {source}' if source else 'Other'

    optimized_df[col_symptom_onset] = optimized_df[col_symptom_onset].apply(map_symptom_onset_optimized)
    print('Unique values (EN):', optimized_df[col_symptom_onset].dropna().unique())
    optimized_df.to_csv('../data/optimized_dataset.csv', index=False)
    print('Updated optimized_df saved to ../data/optimized_dataset.csv')
else:
    print(f'Column not found: {col_symptom_onset}')

Unique values (EN): ['More than 7 years ago' 'Other: i don’t remember' '1-3 years ago'
 'No symptoms' '3-7 years ago' 'Less than 1 year ago'
 'Other: Do not remember' "Don't remember"]
Updated optimized_df saved to ../data/optimized_dataset.csv


In [46]:
# Standardize: Endometriosis time-to-diagnosis column (preserve free-text Other values)
col_endo_diag_time = 'If you have endometriosis or a doctor suspects it: How many years passed between your first symptoms and the first time a doctor mentioned or diagnosed endometriosis?'

if col_endo_diag_time in optimized_df.columns:
    def map_diag_time_optimized(val):
        if pd.isnull(val):
            return None
        original = str(val).strip()
        source = re.sub(r'^(other\s*:\s*)+', '', original, flags=re.IGNORECASE).strip()
        v = source.lower()
        if any(x in v for x in ['have symptoms', 'nobody mentioned', 'niko nije spomenuo', 'doktor nije spomenuo', 'doktor nije pomenuo', 'henüz doktor söylemedi']):
            return 'Have symptoms, not mentioned'
        if any(x in v for x in ['less than 2 years', 'less than two years', 'manje od 2 godine', '2 yıldan az', '2 yildan az']):
            return 'Less than 2 years'
        if any(x in v for x in ['2-5 years', '2–5 years', '2-5 godine', '2–5 godine', '2-5 yıl', '2–5 yıl', '2-5 yil', '2–5 yil']):
            return '2-5 years'
        if any(x in v for x in ['more than 5', 'više od 5', 'vise od 5', '5 yıldan fazla', '5 yildan fazla']):
            return 'More than 5 years'
        if any(x in v for x in ['same year', '0 years', '0 year', 'iste godine', 'aynı yıl', 'ayni yil']):
            return 'Same year as symptoms'
        if any(x in v for x in ['not sure', "don't remember", 'do not remember', 'nisam sigurna', 'ne sjećam se', 'ne sjecam se', 'hatırlamıyorum', 'hatirlamiyorum', 'emin değilim']):
            return 'Not sure'
        return f'Other: {source}' if source else 'Other'

    optimized_df[col_endo_diag_time] = optimized_df[col_endo_diag_time].apply(map_diag_time_optimized)
    print('Unique values (EN):', optimized_df[col_endo_diag_time].dropna().unique())
    optimized_df.to_csv('../data/optimized_dataset.csv', index=False)
    print('Updated optimized_df saved to ../data/optimized_dataset.csv')
else:
    print(f'Column not found: {col_endo_diag_time}')

Unique values (EN): ['Not sure' 'Have symptoms, not mentioned' 'Less than 2 years'
 'More than 5 years' 'Same year as symptoms' '2-5 years']
Updated optimized_df saved to ../data/optimized_dataset.csv


In [47]:
# Standardize: PCOS time-to-diagnosis column
col_pcos_diag_time = 'If you have PCOS or a doctor suspects it: How many years passed between your first symptoms and the first time a doctor mentioned or diagnosed PCOS?'

if col_pcos_diag_time in optimized_df.columns:
    # Reuse same bucket mapping as endometriosis diagnosis delay
    optimized_df[col_pcos_diag_time] = optimized_df[col_pcos_diag_time].apply(map_diag_time_optimized)
    print('Unique values (EN):', optimized_df[col_pcos_diag_time].dropna().unique())
    optimized_df.to_csv('../data/optimized_dataset.csv', index=False)
    print('Updated optimized_df saved to ../data/optimized_dataset.csv')
else:
    print(f'Column not found: {col_pcos_diag_time}')

Unique values (EN): ['Not sure' 'Have symptoms, not mentioned' '2-5 years' 'Less than 2 years'
 'More than 5 years' 'Same year as symptoms']
Updated optimized_df saved to ../data/optimized_dataset.csv


In [48]:
# Standardize: Remedies/strategies tried (multi-select, preserve free-text Other values)
col_remedies = 'Which types of remedies or strategies have you tried for your symptoms? (Select all that apply)\n'

if col_remedies in optimized_df.columns:
    def map_remedies_multiselect(val):
        parts = _split_multiselect(val)
        if not parts:
            return None
        labels = set()
        for p in parts:
            pl = p.lower()
            if any(x in pl for x in ['heat', 'heating', 'warm bath', 'toplota', 'grijanje', 'ısı', 'isi']):
                labels.add('Heat therapy')
                continue
            if any(x in pl for x in ['herbal tea', 'medicinal herb', 'biljni čaj', 'biljni caj', 'bitki çayı', 'bitki cayi']):
                labels.add('Herbal teas/herbs')
                continue
            if any(x in pl for x in ['supplement', 'suplement', 'takviye', 'takviye gıda', 'takviye gida']):
                labels.add('Supplements')
                continue
            if any(x in pl for x in ['stretch', 'mobility', 'istezanje', 'esneme']):
                labels.add('Stretching/mobility')
                continue
            if any(x in pl for x in ['yoga', 'pilates']):
                labels.add('Yoga/Pilates')
                continue
            if any(x in pl for x in ['walking', 'walk', 'šetnja', 'setnja', 'yürüyüş', 'yuruyus']):
                labels.add('Walking')
                continue
            if any(x in pl for x in ['diet', 'nutrition', 'ishrana', 'prehrana', 'beslenme']):
                labels.add('Diet changes')
                continue
            if any(x in pl for x in ['none', 'nemam', 'ne', 'hayır', 'hayir', 'nothing']):
                labels.add('None')
                continue
            if any(x in pl for x in ["don't know", 'not sure', 'nisam sigurna', 'ne znam', 'emin değilim', 'hatırlamıyorum', 'hatirlamiyorum']):
                labels.add("Don't know")
                continue
            if any(x in pl for x in ['other', 'ostalo', 'drugo', 'diğer', 'diger']):
                labels.add(_other_label_from_token(p))
                continue
            labels.add(_other_label_from_token(p))

        real = {'Heat therapy', 'Herbal teas/herbs', 'Supplements', 'Stretching/mobility', 'Yoga/Pilates', 'Walking', 'Diet changes'}
        if labels.intersection(real):
            labels.discard('None')

        ordered = ['Heat therapy', 'Herbal teas/herbs', 'Supplements', 'Stretching/mobility', 'Yoga/Pilates', 'Walking', 'Diet changes', 'None', "Don't know"]
        out = [x for x in ordered if x in labels]
        other_values = sorted([x for x in labels if x.startswith('Other: ')])
        if 'Other' in labels and not other_values:
            out.append('Other')
        out.extend(other_values)
        return '; '.join(out) if out else None

    optimized_df[col_remedies] = optimized_df[col_remedies].apply(map_remedies_multiselect)
    print('Unique values (EN):', optimized_df[col_remedies].dropna().unique())
    optimized_df.to_csv('../data/optimized_dataset.csv', index=False)
    print('Updated optimized_df saved to ../data/optimized_dataset.csv')
else:
    print(f'Column not found: {col_remedies}')

Unique values (EN): ['Heat therapy; Herbal teas/herbs; Supplements; Stretching/mobility; Yoga/Pilates; Walking; Diet changes; Other: More intense physical activity (running; Other: Pain medication (for example ibuprofen; Other: gym; Other: hot water bottle; Other: naproxen; Other: paracetamol); Other: sports); Other: vitamins'
 'Heat therapy; Herbal teas/herbs; Supplements; Other: More intense physical activity (running; Other: Pain medication (for example ibuprofen; Other: gym; Other: hot water bottle; Other: naproxen; Other: paracetamol); Other: sports); Other: vitamins'
 'Heat therapy; Stretching/mobility; Yoga/Pilates; Walking; Diet changes; Other: Pain medication (for example ibuprofen; Other: hot water bottle; Other: naproxen; Other: paracetamol)'
 'Heat therapy; Other: More intense physical activity (running; Other: Pain medication (for example ibuprofen; Other: gym; Other: hot water bottle; Other: naproxen; Other: paracetamol); Other: sports)'
 'Heat therapy; Herbal teas/herbs;

In [49]:
# Standardize: Herbal teas used (multi-select, preserve free-text Other values)
col_herbal_teas = 'Which herbal teas have you used specifically to help with your symptoms? (Select all that apply)\n'

if col_herbal_teas in optimized_df.columns:
    def map_herbal_teas_multiselect(val):
        parts = _split_multiselect(val)
        if not parts:
            return None
        labels = set()
        for p in parts:
            pl = p.lower()
            if any(x in pl for x in ['chamomile', 'kamilica']):
                labels.add('Chamomile')
                continue
            if any(x in pl for x in ['ginger', 'đumbir', 'dumbir', 'zencefil']):
                labels.add('Ginger')
                continue
            if any(x in pl for x in ['cinnamon', 'cimet', 'tarçın', 'tarcin']):
                labels.add('Cinnamon')
                continue
            if any(x in pl for x in ['spearmint', 'zelena metvica']):
                labels.add('Spearmint')
                continue
            if any(x in pl for x in ['peppermint', 'nana', 'menta']):
                labels.add('Peppermint')
                continue
            if any(x in pl for x in ['raspberry leaf', 'list maline']):
                labels.add('Raspberry leaf')
                continue
            if any(x in pl for x in ['fennel', 'komorač', 'komorac', 'rezene']):
                labels.add('Fennel')
                continue
            if any(x in pl for x in ['turmeric', 'kurkuma', 'zerdeçal', 'zerdecal']):
                labels.add('Turmeric')
                continue
            if any(x in pl for x in ['parsley', 'peršun', 'persun', 'maydanoz']):
                labels.add('Parsley')
                continue
            if any(x in pl for x in ['valeriana', 'macina trava', 'kediotu']):
                labels.add('Valeriana officinalis')
                continue
            if any(x in pl for x in ['marigold', 'neven']):
                labels.add('Marigold')
                continue
            if any(x in pl for x in ['st john', 'kantarion', 'sarı kantaron', 'sari kantaron']):
                labels.add('St Johns wort')
                continue
            if any(x in pl for x in ['yarrow', 'achillea', 'kunica', 'hajdučka trava', 'hajducka trava', 'civanpercemi']):
                labels.add('Yarrow')
                continue
            if any(x in pl for x in ['sage', 'salvia', 'kadulja', 'adaçayı', 'adacayi']):
                labels.add('Sage')
                continue
            if any(x in pl for x in ['rosemary', 'ruzmarin', 'biberiye']):
                labels.add('Rosemary')
                continue
            if any(x in pl for x in ['geranium', 'zdravca']):
                labels.add('Geranium')
                continue
            if any(x in pl for x in ['elderflower', 'zova', 'mürver', 'murver']):
                labels.add('Elderflower')
                continue
            if any(x in pl for x in ['none', 'nemam', 'hayır', 'hayir', 'nothing']):
                labels.add('None')
                continue
            if any(x in pl for x in ["don't know", 'not sure', 'nisam sigurna', 'ne znam', 'emin değilim', 'hatırlamıyorum', 'hatirlamiyorum']):
                labels.add("Don't know")
                continue
            if any(x in pl for x in ['other', 'ostalo', 'drugo', 'diğer', 'diger']):
                labels.add(_other_label_from_token(p))
                continue
            labels.add(_other_label_from_token(p))

        tea_labels = {'Chamomile', 'Ginger', 'Cinnamon', 'Spearmint', 'Peppermint', 'Raspberry leaf', 'Fennel', 'Turmeric', 'Parsley', 'Valeriana officinalis', 'Marigold', 'St Johns wort', 'Yarrow', 'Sage', 'Rosemary', 'Geranium', 'Elderflower'}
        if labels.intersection(tea_labels):
            labels.discard('None')

        ordered = ['Chamomile', 'Ginger', 'Cinnamon', 'Spearmint', 'Peppermint', 'Raspberry leaf', 'Fennel', 'Turmeric', 'Parsley', 'Valeriana officinalis', 'Marigold', 'St Johns wort', 'Yarrow', 'Sage', 'Rosemary', 'Geranium', 'Elderflower', 'None', "Don't know"]
        out = [x for x in ordered if x in labels]
        other_values = sorted([x for x in labels if x.startswith('Other: ')])
        if 'Other' in labels and not other_values:
            out.append('Other')
        out.extend(other_values)
        return '; '.join(out) if out else None

    optimized_df[col_herbal_teas] = optimized_df[col_herbal_teas].apply(map_herbal_teas_multiselect)
    print('Unique values (EN):', optimized_df[col_herbal_teas].dropna().unique())
    optimized_df.to_csv('../data/optimized_dataset.csv', index=False)
    print('Updated optimized_df saved to ../data/optimized_dataset.csv')
else:
    print(f'Column not found: {col_herbal_teas}')

Unique values (EN): ['Chamomile; Fennel; Valeriana officinalis; Marigold; St Johns wort; Yarrow; Sage; Rosemary; Elderflower'
 'Chamomile; Peppermint; Raspberry leaf; Fennel; Turmeric; St Johns wort; Yarrow; Sage'
 "None; Don't know" 'Chamomile; Peppermint; Fennel; Other: Vrkuta'
 'Chamomile; Peppermint; Valeriana officinalis'
 'Chamomile; Turmeric; Parsley; Valeriana officinalis; Elderflower'
 'Peppermint' 'Chamomile; Cinnamon'
 "Chamomile; Cinnamon; Peppermint; Don't know" 'Cinnamon' 'Chamomile'
 'Chamomile; Peppermint'
 'Raspberry leaf; Fennel; Valeriana officinalis; Marigold; Yarrow'
 'Chamomile; Ginger' 'Chamomile; Spearmint'
 'Spearmint; Peppermint; St Johns wort; Sage' 'Chamomile; Turmeric'
 'Chamomile; Ginger; Cinnamon; Spearmint; Peppermint; Turmeric'
 'Chamomile; Ginger; Cinnamon; Peppermint; Raspberry leaf; Rosemary'
 'Chamomile; Ginger; Peppermint; Raspberry leaf; Fennel; Turmeric'
 'Chamomile; Cinnamon; Peppermint; Other: Green Tea'
 'Chamomile; Ginger; Cinnamon; Parsley; 

In [50]:
# Standardize: Supplements used regularly (multi-select, preserve free-text Other values)
col_supplements = 'Which supplements have you used regularly (at least 1–2 months) for your symptoms? (Select all that apply)'

if col_supplements in optimized_df.columns:
    def map_supplements_multiselect(val):
        parts = _split_multiselect(val)
        if not parts:
            return None
        labels = set()
        for p in parts:
            pl = p.lower()
            if any(x in pl for x in ['myo', 'inositol', 'mio-inozitol']):
                labels.add('Myo-inositol')
                continue
            if any(x in pl for x in ['d-chiro', 'd chiro']):
                labels.add('D-chiro-inositol')
                continue
            if any(x in pl for x in ['berberine', 'berberin']):
                labels.add('Berberine')
                continue
            if any(x in pl for x in ['n-acetylcysteine', 'n acetylcysteine', 'nac']):
                labels.add('N-acetylcysteine (NAC)')
                continue
            if any(x in pl for x in ['magnesium', 'magnezij', 'magnezyum']):
                labels.add('Magnesium')
                continue
            if any(x in pl for x in ['zinc', 'cink', 'çinko', 'cinko']):
                labels.add('Zinc')
                continue
            if any(x in pl for x in ['vitamin d', 'vit d']):
                labels.add('Vitamin D')
                continue
            if any(x in pl for x in ['omega-3', 'omega 3', 'omega3']):
                labels.add('Omega-3')
                continue
            if any(x in pl for x in ['none', 'nemam', 'hayır', 'hayir', 'nothing']):
                labels.add('None')
                continue
            if any(x in pl for x in ["don't know", 'not sure', 'nisam sigurna', 'ne znam', 'emin değilim', 'hatırlamıyorum', 'hatirlamiyorum']):
                labels.add("Don't know")
                continue
            if any(x in pl for x in ['other', 'ostalo', 'drugo', 'diğer', 'diger']):
                labels.add(_other_label_from_token(p))
                continue
            labels.add(_other_label_from_token(p))

        supp_labels = {'Myo-inositol', 'D-chiro-inositol', 'Berberine', 'N-acetylcysteine (NAC)', 'Magnesium', 'Zinc', 'Vitamin D', 'Omega-3'}
        if labels.intersection(supp_labels):
            labels.discard('None')

        ordered = ['Myo-inositol', 'D-chiro-inositol', 'Berberine', 'N-acetylcysteine (NAC)', 'Magnesium', 'Zinc', 'Vitamin D', 'Omega-3', 'None', "Don't know"]
        out = [x for x in ordered if x in labels]
        other_values = sorted([x for x in labels if x.startswith('Other: ')])
        if 'Other' in labels and not other_values:
            out.append('Other')
        out.extend(other_values)
        return '; '.join(out) if out else None

    optimized_df[col_supplements] = optimized_df[col_supplements].apply(map_supplements_multiselect)
    print('Unique values (EN):', optimized_df[col_supplements].dropna().unique())
    optimized_df.to_csv('../data/optimized_dataset.csv', index=False)
    print('Updated optimized_df saved to ../data/optimized_dataset.csv')
else:
    print(f'Column not found: {col_supplements}')

Unique values (EN): ['Magnesium; Vitamin D; Other: Omega‑3 (fish oil or similar)'
 "None; Don't know" 'Magnesium' 'Vitamin D'
 'Vitamin D; Other: Omega‑3 (fish oil or similar); Other: iron'
 'Magnesium; Other: Omega‑3 (fish oil or similar)'
 'Magnesium; Vitamin D; Other: iron supplement' 'Magnesium; Vitamin D'
 'N-acetylcysteine (NAC); Magnesium; Vitamin D; Other: Omega‑3 (fish oil or similar)'
 'Zinc; Vitamin D; Other: Omega‑3 (fish oil or similar)'
 'Magnesium; Zinc; Vitamin D; Other: Omega‑3 (fish oil or similar)'
 'Zinc; Vitamin D; Other: Omega‑3 (riblje ulje ili slično )'
 "Don't know; Other: Nijedan" 'Magnesium; Other: Vitamin C'
 'Magnesium; Zinc'
 'Magnesium; Zinc; Vitamin D; Other: Omega‑3 (riblje ulje ili slično )'
 'Magnesium; Vitamin D; Other: Omega‑3 (riblje ulje ili slično )'
 'Other: Lekolid' 'Magnesium; Zinc; Vitamin D'
 "Magnesium; Vitamin D; Don't know; Other: Nijedan; Other: Omega‑3 (riblje ulje ili slično )"
 'Magnesium; Zinc; Other: Omega‑3 (riblje ulje ili slično 

In [51]:
# Standardize: Physical activity types used for symptoms (multi-select, preserve free-text Other values)
col_activity = 'Which types of physical activity have you used specifically to help your symptoms? (Select all that apply)'

if col_activity in optimized_df.columns:
    def map_activity_multiselect(val):
        parts = _split_multiselect(val)
        if not parts:
            return None
        labels = set()
        for p in parts:
            pl = p.lower()
            if any(x in pl for x in ['gentle stretching', 'mobility', 'istezanje', 'esneme']):
                labels.add('Gentle stretching/mobility')
                continue
            if any(x in pl for x in ['yoga', 'pilates']):
                labels.add('Yoga/Pilates')
                continue
            if any(x in pl for x in ['walking', 'walk', 'šetnja', 'setnja', 'yürüyüş', 'yuruyus']):
                labels.add('Walking')
                continue
            if any(x in pl for x in ['running', 'jogging', 'trčanje', 'trcanje', 'koşu', 'kosu']):
                labels.add('Running/jogging')
                continue
            if any(x in pl for x in ['weight training', 'strength training', 'tegovi', 'ağırlık', 'agirlik']):
                labels.add('Weight training')
                continue
            if any(x in pl for x in ['cardio', 'kardio']):
                labels.add('Cardio')
                continue
            if any(x in pl for x in ['swimming', 'water exercise', 'plivanje', 'yüzme', 'yuzme']):
                labels.add('Swimming/water exercise')
                continue
            if any(x in pl for x in ['pelvic floor', 'physiotherapy', 'fizikalna', 'fizyoterapi']):
                labels.add('Pelvic floor/physiotherapy')
                continue
            if any(x in pl for x in ['none', 'nemam', 'hayır', 'hayir', 'nothing']):
                labels.add('None')
                continue
            if any(x in pl for x in ["don't know", 'not sure', 'nisam sigurna', 'ne znam', 'emin değilim', 'hatırlamıyorum', 'hatirlamiyorum']):
                labels.add("Don't know")
                continue
            if any(x in pl for x in ['other', 'ostalo', 'drugo', 'diğer', 'diger']):
                labels.add(_other_label_from_token(p))
                continue
            labels.add(_other_label_from_token(p))

        activity_labels = {'Gentle stretching/mobility', 'Yoga/Pilates', 'Walking', 'Running/jogging', 'Weight training', 'Cardio', 'Swimming/water exercise', 'Pelvic floor/physiotherapy'}
        if labels.intersection(activity_labels):
            labels.discard('None')

        ordered = ['Gentle stretching/mobility', 'Yoga/Pilates', 'Walking', 'Running/jogging', 'Weight training', 'Cardio', 'Swimming/water exercise', 'Pelvic floor/physiotherapy', 'None', "Don't know"]
        out = [x for x in ordered if x in labels]
        other_values = sorted([x for x in labels if x.startswith('Other: ')])
        if 'Other' in labels and not other_values:
            out.append('Other')
        out.extend(other_values)
        return '; '.join(out) if out else None

    optimized_df[col_activity] = optimized_df[col_activity].apply(map_activity_multiselect)
    print('Unique values (EN):', optimized_df[col_activity].dropna().unique())
    optimized_df.to_csv('../data/optimized_dataset.csv', index=False)
    print('Updated optimized_df saved to ../data/optimized_dataset.csv')
else:
    print(f'Column not found: {col_activity}')

Unique values (EN): ['Gentle stretching/mobility; Yoga/Pilates; Pelvic floor/physiotherapy'
 'Gentle stretching/mobility; Walking; Weight training; Cardio'
 'Gentle stretching/mobility'
 'Gentle stretching/mobility; Running/jogging; Cardio'
 'Gentle stretching/mobility; Walking'
 'None; Other: I have not used physical activity as a remedy'
 'Gentle stretching/mobility; Running/jogging; Weight training' 'Walking'
 'Walking; Swimming/water exercise' 'Weight training; Cardio'
 'Gentle stretching/mobility; Yoga/Pilates; Walking; Weight training; Cardio; Pelvic floor/physiotherapy; Other: Group classes (for example aerobics; Other: Zumba; Other: dance; Other: etc.)'
 'Gentle stretching/mobility; Walking; Swimming/water exercise'
 'Gentle stretching/mobility; Yoga/Pilates; Walking; Running/jogging'
 'Gentle stretching/mobility; Pelvic floor/physiotherapy'
 'Gentle stretching/mobility; Yoga/Pilates; Walking; Running/jogging; Weight training'
 'Yoga/Pilates; Weight training; Cardio; Pelvic flo

In [52]:
# Standardize: Diet changes for endometriosis/PCOS symptoms (multi-select, preserve free-text Other values)
col_diet = 'Have you changed your diet specifically to help with endometriosis / PCOS symptoms? (Select all that apply)'

if col_diet in optimized_df.columns:
    def map_diet_multiselect(val):
        parts = _split_multiselect(val)
        if not parts:
            return None
        labels = set()
        for p in parts:
            pl = p.lower()
            if any(x in pl for x in ['higher protein', 'high protein', 'više proteina', 'vise proteina', 'yüksek protein', 'yuksek protein']):
                labels.add('Higher protein diet')
                continue
            if any(x in pl for x in ['lower-carb', 'lower carb', 'low carb', 'manje ugljikohidrata', 'düşük karbonhidrat', 'dusuk karbonhidrat']):
                labels.add('Lower-carb diet')
                continue
            if any(x in pl for x in ['keto']):
                labels.add('Keto diet')
                continue
            if any(x in pl for x in ['anti-inflammatory', 'anti inflammatory', 'protuupalna', 'antiinflamatuar']):
                labels.add('Anti-inflammatory diet')
                continue
            if any(x in pl for x in ['reduced sugar', 'less sugar', 'manje šećera', 'manje secera', 'az şeker', 'az seker']):
                labels.add('Reduced sugar/sweets')
                continue
            if any(x in pl for x in ['reduced processed', 'fast food', 'manje procesirane', 'işlenmiş gıda', 'islenmis gida']):
                labels.add('Reduced processed/fast food')
                continue
            if any(x in pl for x in ['reduced gluten', 'less gluten', 'bez glutena', 'glutensiz']):
                labels.add('Reduced gluten')
                continue
            if any(x in pl for x in ['reduced dairy', 'less dairy', 'manje mliječnih', 'manje mlijecnih', 'süt ürünlerini azalt', 'sut urunlerini azalt']):
                labels.add('Reduced dairy')
                continue
            if any(x in pl for x in ['vegetarian', 'vegan', 'vegetarijan', 'vejetaryen']):
                labels.add('Vegetarian/vegan')
                continue
            if any(x in pl for x in ['intermittent fasting', 'time-restricted', 'isprekidani post', 'aralıklı oruç', 'aralikli oruc']):
                labels.add('Intermittent fasting/time-restricted eating')
                continue
            if any(x in pl for x in ['none', 'nemam', 'hayır', 'hayir', 'nothing']):
                labels.add('None')
                continue
            if any(x in pl for x in ["don't know", 'not sure', 'nisam sigurna', 'ne znam', 'emin değilim', 'hatırlamıyorum', 'hatirlamiyorum']):
                labels.add("Don't know")
                continue
            if any(x in pl for x in ['other', 'ostalo', 'drugo', 'diğer', 'diger']):
                labels.add(_other_label_from_token(p))
                continue
            labels.add(_other_label_from_token(p))

        diet_labels = {'Higher protein diet', 'Lower-carb diet', 'Keto diet', 'Anti-inflammatory diet', 'Reduced sugar/sweets', 'Reduced processed/fast food', 'Reduced gluten', 'Reduced dairy', 'Vegetarian/vegan', 'Intermittent fasting/time-restricted eating'}
        if labels.intersection(diet_labels):
            labels.discard('None')

        ordered = ['Higher protein diet', 'Lower-carb diet', 'Keto diet', 'Anti-inflammatory diet', 'Reduced sugar/sweets', 'Reduced processed/fast food', 'Reduced gluten', 'Reduced dairy', 'Vegetarian/vegan', 'Intermittent fasting/time-restricted eating', 'None', "Don't know"]
        out = [x for x in ordered if x in labels]
        other_values = sorted([x for x in labels if x.startswith('Other: ')])
        if 'Other' in labels and not other_values:
            out.append('Other')
        out.extend(other_values)
        return '; '.join(out) if out else None

    optimized_df[col_diet] = optimized_df[col_diet].apply(map_diet_multiselect)
    print('Unique values (EN):', optimized_df[col_diet].dropna().unique())
    optimized_df.to_csv('../data/optimized_dataset.csv', index=False)
    print('Updated optimized_df saved to ../data/optimized_dataset.csv')
else:
    print(f'Column not found: {col_diet}')

Unique values (EN): ['Higher protein diet; Reduced sugar/sweets; Reduced processed/fast food; Other: Anti‑inflammatory diet (for example focusing on whole foods; Other: fruits; Other: healthy fats); Other: sweets; Other: vegetables'
 'None; Other: I have not changed my diet for these symptoms'
 'Lower-carb diet; Reduced gluten; Other: low‑carb diet'
 'Reduced sugar/sweets; Reduced processed/fast food; Other: Anti‑inflammatory diet (for example focusing on whole foods; Other: fruits; Other: healthy fats); Other: sweets; Other: vegetables'
 'Reduced gluten; Reduced dairy; Other: I have not changed my diet for these symptoms'
 'None; Other: I have not changed my diet for these symptoms; Other: stopped drinking coffee'
 'Other: Anti‑inflammatory diet (for example focusing on whole foods; Other: fruits; Other: healthy fats); Other: vegetables'
 'Vegetarian/vegan; Other: but it just helps me; Other: i love it; Other: İ am a vegeterian not related to any symptoms'
 'Other: I dont have it' 'Hi

In [53]:
# Standardize: cycle-related mood swings/anxiety/depression frequency (preserve free-text Other values)
mood_col_candidates = [
    'In the last 6 months, how often have you had mood swings, anxiety or depression clearly related to your cycle? ',
    'In the last 6 months, how often have you had mood swings, anxiety or depression clearly related to your cycle?'
]

mood_col = next((c for c in mood_col_candidates if c in optimized_df.columns), None)
if mood_col is None:
    col_norm = {c.strip().lower(): c for c in optimized_df.columns}
    mood_col = col_norm.get('in the last 6 months, how often have you had mood swings, anxiety or depression clearly related to your cycle?')

if mood_col is None:
    print('Mood-cycle column not found in optimized_df')
else:
    def map_mood_cycle_frequency(val):
        if pd.isnull(val):
            return None
        original = str(val).strip()
        source = re.sub(r'^(other\s*:\s*)+', '', original, flags=re.IGNORECASE).strip()
        v = source.lower()

        if any(x in v for x in ['never', 'nikad', 'hiçbir zaman', 'hicbir zaman']):
            return 'Never'
        if any(x in v for x in ['rarely', 'rijetko', 'nadiren']):
            return 'Rarely'
        if any(x in v for x in ['some cycles', 'poneki ciklus', 'bazı döngü', 'bazi dongu', 'some months']):
            return 'Some cycles'
        if any(x in v for x in ['most cycles', 'većinu ciklusa', 'vecinu ciklusa', 'çoğu döngü', 'cogu dongu']):
            return 'Most cycles'
        if any(x in v for x in ['every cycle', 'svaki ciklus', 'her döngü', 'her dongu']):
            return 'Every cycle'
        if any(x in v for x in ["don't know", 'do not know', 'not sure', 'nisam sigurna', 'ne znam', 'emin değilim', 'hatırlamıyorum', 'hatirlamiyorum']):
            return "Don't know"

        return f'Other: {source}' if source else 'Other'

    optimized_df[mood_col] = optimized_df[mood_col].apply(map_mood_cycle_frequency)
    print('Unique values (EN):', optimized_df[mood_col].dropna().unique())
    optimized_df.to_csv('../data/optimized_dataset.csv', index=False)
    print('Updated optimized_df saved to ../data/optimized_dataset.csv')

Unique values (EN): ['Some cycles' 'Every cycle' 'Most cycles' 'Rarely' 'Never']
Updated optimized_df saved to ../data/optimized_dataset.csv


In [54]:
# Audit: print all columns containing 'Other' and the exact 'Other' contents
import re

other_cols = []

for col in optimized_df.columns:
    s = optimized_df[col].dropna().astype(str)
    # Match plain 'Other' or semicolon-separated tokens with 'Other' / 'Other: ...'
    mask = s.str.contains(r'(^|;\s*)other(\s*:\s*.*)?($|\s*;)', case=False, regex=True)
    if mask.any():
        other_cols.append(col)
        values = s[mask].unique().tolist()

        extracted = []
        for v in values:
            parts = [p.strip() for p in v.split(';')]
            for p in parts:
                if re.match(r'^other\b', p, flags=re.IGNORECASE):
                    extracted.append(p)

        unique_extracted = sorted(set(extracted), key=lambda x: x.lower())

        print(f"\nColumn: {col}")
        print(f"Rows with Other: {mask.sum()}")
        if unique_extracted:
            print('Other contents:')
            for item in unique_extracted:
                print(f"  - {item}")
        else:
            print('Other contents: (plain Other only)')

print('\n' + '=' * 70)
print(f'Total columns containing Other: {len(other_cols)}')


Column: Do you experience pain or burning during sexual intercourse (dyspareunia)?
Rows with Other: 2
Other contents:
  - Other

Column: Do you experience pain after sexual intercourse?
Rows with Other: 2
Other contents:
  - Other

Column: How often do you have noticeable bloating or “endo belly”?
Rows with Other: 63
Other contents:
  - Other

Column: Around your period (from 3 days before bleeding until the last day of bleeding), how often do you usually have diarrhea?
Rows with Other: 18
Other contents:
  - Other

Column: Around your period (from 3 days before bleeding until the last day of bleeding), how often do you usually have constipation or very hard stools?
Rows with Other: 11
Other contents:
  - Other

Column: How often do you feel very tired or exhausted, not improved by sleep? 
Rows with Other: 14
Other contents:
  - Other

Column: Have you been told you have any of the following? (select all that apply)
Rows with Other: 83
Other contents:
  - Other: ali bez sindroma polic

/var/folders/lw/phzf3tkd1lv_x4p28fxx1btw0000gn/T/ipykernel_87054/847923954.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = s.str.contains(r'(^|;\s*)other(\s*:\s*.*)?($|\s*;)', case=False, regex=True)
/var/folders/lw/phzf3tkd1lv_x4p28fxx1btw0000gn/T/ipykernel_87054/847923954.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = s.str.contains(r'(^|;\s*)other(\s*:\s*.*)?($|\s*;)', case=False, regex=True)
/var/folders/lw/phzf3tkd1lv_x4p28fxx1btw0000gn/T/ipykernel_87054/847923954.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = s.str.contains(r'(^|;\s*)other(\s*:\s*.*)?($|\s*;)', case=False, regex=True)
/var/folders/lw/phzf3tkd1lv_x4p28fxx1btw0000gn/T/ipykernel_87054/847923954.py:9: UserWarning: This p

### Standardize 'How often do you have noticeable bloating or “endo belly”?' column to English

In [55]:
# Display standardized bloating column values
bloating_candidates = [
    'How often do you have noticeable bloating or "endo belly"?',
    'How often do you have noticeable bloating or "endo belly"?'
]

bloating_col = next((c for c in bloating_candidates if c in optimized_df.columns), None)
if bloating_col is None:
    print('Bloating column not found in optimized_df')
else:
    print('Unique standardized values in bloating column (EN):', optimized_df[bloating_col].dropna().unique())

Bloating column not found in optimized_df


### Standardize multi-select columns to English

In [56]:
# Display standardized multi-select columns
multi_select_cols = [
    'Have you been told you have any of the following? (select all that apply)',
    'Do you have any of these hormone‑related features? (select all that apply)',
    'Have you ever had a pelvic ultrasound? If yes, were you ever told any of these findings? (select all that apply)'
]

for col_name in multi_select_cols:
    if col_name in optimized_df.columns:
        print(f'\nUnique standardized values in {col_name}:')
        print(optimized_df[col_name].dropna().unique())
    else:
        print(f'Column not found: {col_name}')


Unique standardized values in Have you been told you have any of the following? (select all that apply):
['None'
 'Endometriosis; IBS; Thyroid disorder; Other: chronic pain condition'
 'Endometriosis' 'PCOS; Other: ovarian cysts'
 'Endometriosis; Other: ovarian cysts'
 'IBS; Thyroid disorder; Other: inflammatory bowel disease (ibd); Other: ovarian cysts'
 'PCOS' 'Endometriosis; Adenomyosis' 'PCOS; Thyroid disorder'
 'Thyroid disorder; Other: chronic pain condition'
 'Endometriosis; PCOS; Adenomyosis; Other: ovarian cysts; Other: pelvic inflammatory disease'
 'Endometriosis; Thyroid disorder; Other: chronic pain condition'
 'PCOS; Other: chocolate cyst; Other: endometrioma; Other: ovarian cysts'
 'Endometriosis; Fibroids' 'IBS; Thyroid disorder' 'Other: ovarian cysts'
 'Other: chronic pain condition'
 'Other: ciste na jajnicima; Other: endomentrioza' 'Thyroid disorder'
 'Fibroids; IBS; Thyroid disorder; Other: autoimuna bolest (npr. lupus; Other: ciste na jajnicima; Other: reumatoidni 

In [57]:
# Print unique values for the 4 target columns
cols_to_check = [
    'Have you been told you have any of the following? (select all that apply)',
    'Do you have any of these hormone‑related features? (select all that apply)',
    'Have you ever had a pelvic ultrasound? If yes, were you ever told any of these findings? (select all that apply)',
    'How long ago did your symptoms (pain, cycle changes or other main problems) first start?'
]

for col_name in cols_to_check:
    if col_name in optimized_df.columns:
        print(f'\n=== {col_name} ===')
        print(f'Unique count: {optimized_df[col_name].nunique()}')
        print('Unique values:')
        unique_vals = optimized_df[col_name].dropna().unique()
        for i, val in enumerate(unique_vals, 1):
            print(f'{i}. {val}')
    else:
        print(f'Column NOT found: {col_name}')


=== Have you been told you have any of the following? (select all that apply) ===
Unique count: 63
Unique values:
1. None
2. Endometriosis; IBS; Thyroid disorder; Other: chronic pain condition
3. Endometriosis
4. PCOS; Other: ovarian cysts
5. Endometriosis; Other: ovarian cysts
6. IBS; Thyroid disorder; Other: inflammatory bowel disease (ibd); Other: ovarian cysts
7. PCOS
8. Endometriosis; Adenomyosis
9. PCOS; Thyroid disorder
10. Thyroid disorder; Other: chronic pain condition
11. Endometriosis; PCOS; Adenomyosis; Other: ovarian cysts; Other: pelvic inflammatory disease
12. Endometriosis; Thyroid disorder; Other: chronic pain condition
13. PCOS; Other: chocolate cyst; Other: endometrioma; Other: ovarian cysts
14. Endometriosis; Fibroids
15. IBS; Thyroid disorder
16. Other: ovarian cysts
17. Other: chronic pain condition
18. Other: ciste na jajnicima; Other: endomentrioza
19. Thyroid disorder
20. Fibroids; IBS; Thyroid disorder; Other: autoimuna bolest (npr. lupus; Other: ciste na jaj

In [58]:
# Standardize: 'Have you been told you have any of the following?' (multi-select)
target_col = 'Have you been told you have any of the following? (select all that apply)'
if target_col in optimized_df.columns:
    print(f'Column found: {target_col}')
    print('Already standardized. Unique values:')
    print(optimized_df[target_col].dropna().unique())
else:
    print(f'Column NOT found: {target_col}')

Column found: Have you been told you have any of the following? (select all that apply)
Already standardized. Unique values:
['None'
 'Endometriosis; IBS; Thyroid disorder; Other: chronic pain condition'
 'Endometriosis' 'PCOS; Other: ovarian cysts'
 'Endometriosis; Other: ovarian cysts'
 'IBS; Thyroid disorder; Other: inflammatory bowel disease (ibd); Other: ovarian cysts'
 'PCOS' 'Endometriosis; Adenomyosis' 'PCOS; Thyroid disorder'
 'Thyroid disorder; Other: chronic pain condition'
 'Endometriosis; PCOS; Adenomyosis; Other: ovarian cysts; Other: pelvic inflammatory disease'
 'Endometriosis; Thyroid disorder; Other: chronic pain condition'
 'PCOS; Other: chocolate cyst; Other: endometrioma; Other: ovarian cysts'
 'Endometriosis; Fibroids' 'IBS; Thyroid disorder' 'Other: ovarian cysts'
 'Other: chronic pain condition'
 'Other: ciste na jajnicima; Other: endomentrioza' 'Thyroid disorder'
 'Fibroids; IBS; Thyroid disorder; Other: autoimuna bolest (npr. lupus; Other: ciste na jajnicima; 

In [59]:
# Standardize: 'Do you have any of these hormone-related features?' (multi-select)
hormone_col = 'Do you have any of these hormone‑related features? (select all that apply)'
if hormone_col in optimized_df.columns:
    print(f'Column found: {hormone_col}')
    print('Already standardized. Unique values:')
    print(optimized_df[hormone_col].dropna().unique())
else:
    print(f'Column NOT found: {hormone_col}')

Column found: Do you have any of these hormone‑related features? (select all that apply)
Already standardized. Unique values:
['None'
 'Acne/oily skin; Hair thinning/hair loss; Other: armpits; Other: folds)'
 'Acne/oily skin; Other: Excess hair growth (chin; Other: abdomen; Other: chest; Other: thighs); Other: upper lip'
 'Acne/oily skin'
 'None; Other: Excess hair growth (chin; Other: abdomen; Other: armpits; Other: chest; Other: folds); Other: thighs); Other: upper lip'
 'Hair thinning/hair loss'
 'Weight gain/difficulty losing weight; Other: Difficulty losing weight'
 'Weight gain/difficulty losing weight; Other: Difficulty losing weight; Other: Excess hair growth (chin; Other: abdomen; Other: chest; Other: thighs); Other: upper lip'
 'Other: Difficulty losing weight'
 'Hair thinning/hair loss; Weight gain/difficulty losing weight; Other: Difficulty losing weight; Other: Excess hair growth (chin; Other: abdomen; Other: chest; Other: thighs); Other: upper lip'
 'Acne/oily skin; Other

In [60]:
# Standardize: 'Have you ever had a pelvic ultrasound findings?' (multi-select)
ultrasound_col = 'Have you ever had a pelvic ultrasound? If yes, were you ever told any of these findings? (select all that apply)'
if ultrasound_col in optimized_df.columns:
    print(f'Column found: {ultrasound_col}')
    print('Already standardized. Unique values:')
    print(optimized_df[ultrasound_col].dropna().unique())
else:
    print(f'Column NOT found: {ultrasound_col}')

Column found: Have you ever had a pelvic ultrasound? If yes, were you ever told any of these findings? (select all that apply)
Already standardized. Unique values:
['No abnormal findings'
 "Other: I don't remember; Other: I never had an ultrasound"
 'Ovarian cyst; Other: Polycystic‑looking ovaries (described as ‘polycystic’ or ‘many small follicles’); Other: Thickened endometrium'
 'Endometrioma/ovarian cyst; Other: Thickened endometrium' 'Ovarian cyst'
 'Endometrioma/ovarian cyst; Adenomyosis'
 'Other: Polycystic‑looking ovaries (described as ‘polycystic’ or ‘many small follicles’)'
 'Endometrioma/ovarian cyst; Ovarian cyst; Adenomyosis; Other: Fluid in the pelvis; Other: Polycystic‑looking ovaries (described as ‘polycystic’ or ‘many small follicles’); Other: Thickened endometrium'
 'Endometrioma/ovarian cyst'
 'Endometrioma/ovarian cyst; Ovarian cyst; Other: Fluid in the pelvis; Other: Polycystic‑looking ovaries (described as ‘polycystic’ or ‘many small follicles’); Other: Thickened 

In [61]:
# Standardize: 'How long ago did your symptoms first start?' (frequency/time-based)
symptom_col = 'How long ago did your symptoms (pain, cycle changes or other main problems) first start?'
if symptom_col in optimized_df.columns:
    print(f'Column found: {symptom_col}')
    print('Already standardized. Unique values:')
    unique_vals = optimized_df[symptom_col].dropna().unique()
    print(f'Unique count: {len(unique_vals)}')
    for val in sorted(unique_vals):
        print(f'  - {val}')
else:
    print(f'Column NOT found: {symptom_col}')

Column found: How long ago did your symptoms (pain, cycle changes or other main problems) first start?
Already standardized. Unique values:
Unique count: 8
  - 1-3 years ago
  - 3-7 years ago
  - Don't remember
  - Less than 1 year ago
  - More than 7 years ago
  - No symptoms
  - Other: Do not remember
  - Other: i don’t remember


In [62]:
# Save standardized optimized_df to CSV
optimized_df.to_csv('../data/optimized_dataset.csv', index=False)
print('✓ Updated optimized_df saved to ../data/optimized_dataset.csv')
print(f'Total rows: {len(optimized_df)}')
print(f'Total columns: {len(optimized_df.columns)}')

✓ Updated optimized_df saved to ../data/optimized_dataset.csv
Total rows: 175
Total columns: 100


In [63]:

col_ultrasound_findings = 'Have you ever had a pelvic ultrasound? If yes, were you ever told any of these findings? (select all that apply)'
if col_ultrasound_findings in optimized_df.columns:
    print(f'Column found: {col_ultrasound_findings}')
    print('Unique values:')
    unique_vals = optimized_df[col_ultrasound_findings].dropna().unique()
    for val in sorted(unique_vals):
        print(f'  - {val}')
else:
    print(f'Column NOT found: {col_ultrasound_findings}')

Column found: Have you ever had a pelvic ultrasound? If yes, were you ever told any of these findings? (select all that apply)
Unique values:
  - Adenomyosis; Other: Cista (ciste) na jajniku; Other: Policističan izgled jajnika (opisano kao „policistični“ ili „mnogo malih folikula“); Other: Slobodna tečnost u karlici; Other: Zadebljan endometrij (sluznica maternice)
  - Adenomyosis; Other: Cista (ciste) na jajniku; Other: Policističan izgled jajnika (opisano kao „policistični“ ili „mnogo malih folikula“); Other: Zadebljan endometrij (sluznica maternice)
  - Adenomyosis; Other: Cista (ciste) na jajniku; Other: Zadebljan endometrij (sluznica maternice)
  - Adenomyosis; Other: Retrocervikalno se nalaze čvorovi i na sakrouterinim ligamentima
  - Endometrioma/ovarian cyst
  - Endometrioma/ovarian cyst; Adenomyosis
  - Endometrioma/ovarian cyst; Adenomyosis; Other: Cista (ciste) na jajniku; Other: Policističan izgled jajnika (opisano kao „policistični“ ili „mnogo malih folikula“); Other: Zade

In [64]:
# Standardize and translate pelvic ultrasound findings to English
ultrasound_col = 'Have you ever had a pelvic ultrasound? If yes, were you ever told any of these findings? (select all that apply)'

def standardize_ultrasound_english(val):
    if pd.isnull(val):
        return None
    
    text = str(val).strip()
    if text.lower() in ['', 'nan', 'none']:
        return None
    
    # Translation mappings
    translations = {
        'Cista (ciste) na jajniku': 'Ovarian cyst',
        'Policističan izgled jajnika': 'Polycystic-looking ovaries',
        'Policističan izgled jajnika (opisano kao „policistični" ili „mnogo malih folikula")': 'Polycystic-looking ovaries',
        'Zadebljan endometrij': 'Thickened endometrium',
        'Zadebljan endometrij (sluznica maternice)': 'Thickened endometrium',
        'Slobodna tečnost u karlici': 'Fluid in the pelvis',
        'Polikistik görünümlü overler': 'Polycystic-looking ovaries',
        'Over kisti(leri)': 'Ovarian cyst',
        'Ne sjećam se': "Don't remember",
        'Nikada nisam radila ultrazvuk': 'Never had ultrasound',
        'Sve je bilo uredu': 'Everything was fine',
        'Ranica na grlicu maternoce aktivna i cin 3': 'Cervical wound (CIN 3)',
        'Srcolika maternica': 'Heart-shaped uterus',
    }
    
    # Split by semicolon and process each part
    parts = [p.strip() for p in text.split(';')]
    translated_parts = []
    
    for part in parts:
        if part.startswith('Other: '):
            other_text = part[7:].strip()  # Remove 'Other: ' prefix
            # Try to translate the other text
            translated = translations.get(other_text, other_text)
            translated_parts.append(translated)
        else:
            # Already standardized labels
            translated_parts.append(part)
    
    # Remove duplicates while preserving order
    unique_parts = []
    seen = set()
    for part in translated_parts:
        if part not in seen:
            unique_parts.append(part)
            seen.add(part)
    
    return '; '.join(unique_parts) if unique_parts else None

if ultrasound_col in optimized_df.columns:
    optimized_df[ultrasound_col] = optimized_df[ultrasound_col].apply(standardize_ultrasound_english)
    print(f'✓ Standardized {ultrasound_col} to English')
    print('\nUnique standardized values:')
    for val in sorted(optimized_df[ultrasound_col].dropna().unique()):
        print(f'  - {val}')
else:
    print(f'Column NOT found: {ultrasound_col}')

✓ Standardized Have you ever had a pelvic ultrasound? If yes, were you ever told any of these findings? (select all that apply) to English

Unique standardized values:
  - Adenomyosis; Ovarian cyst; Policističan izgled jajnika (opisano kao „policistični“ ili „mnogo malih folikula“); Fluid in the pelvis; Thickened endometrium
  - Adenomyosis; Ovarian cyst; Policističan izgled jajnika (opisano kao „policistični“ ili „mnogo malih folikula“); Thickened endometrium
  - Adenomyosis; Ovarian cyst; Thickened endometrium
  - Adenomyosis; Retrocervikalno se nalaze čvorovi i na sakrouterinim ligamentima
  - Cervical wound (CIN 3)
  - Don't remember; Never had ultrasound
  - Endometrioma/ovarian cyst
  - Endometrioma/ovarian cyst; Adenomyosis
  - Endometrioma/ovarian cyst; Adenomyosis; Ovarian cyst; Policističan izgled jajnika (opisano kao „policistični“ ili „mnogo malih folikula“); Thickened endometrium
  - Endometrioma/ovarian cyst; Adenomyosis; Thickened endometrium
  - Endometrioma/ovarian cys

In [65]:
# Save standardized and translated data to CSV
optimized_df.to_csv('../data/optimized_dataset.csv', index=False)
print('✓ Updated optimized_df with translated ultrasound findings saved to ../data/optimized_dataset.csv')
print(f'Total rows: {len(optimized_df)}')
print(f'Total columns: {len(optimized_df.columns)}')

✓ Updated optimized_df with translated ultrasound findings saved to ../data/optimized_dataset.csv
Total rows: 175
Total columns: 100


In [66]:
col_ultrasound_findings = 'Have you been told you have any of the following? (select all that apply)'
if col_ultrasound_findings in optimized_df.columns:
    print(f'Column found: {col_ultrasound_findings}')
    print('Unique values:')
    unique_vals = optimized_df[col_ultrasound_findings].dropna().unique()
    for val in sorted(unique_vals):
        print(f'  - {val}')
else:
    print(f'Column NOT found: {col_ultrasound_findings}')

Column found: Have you been told you have any of the following? (select all that apply)
Unique values:
  - Adenomyosis; Fibroids; Other: autoimuna bolest (npr. lupus; Other: ciste na jajnicima; Other: endomentrioza; Other: hronično stanje bola; Other: reumatoidni artritis); Other: upalna bolest karlice
  - Adenomyosis; Fibroids; Other: ciste na jajnicima; Other: hronično stanje bola
  - Adenomyosis; Fibroids; Other: endomentrioza
  - Adenomyosis; Other: ciste na jajnicima; Other: endomentrioza
  - Adenomyosis; Other: ciste na jajnicima; Other: endomentrioza; Other: hidradenitis suppurativa
  - Adenomyosis; Other: endomentrioza
  - Adenomyosis; Thyroid disorder; Other: endomentrioza
  - Endometriosis
  - Endometriosis; Adenomyosis
  - Endometriosis; Fibroids
  - Endometriosis; IBS; Thyroid disorder; Other: chronic pain condition
  - Endometriosis; Other: ovarian cysts
  - Endometriosis; PCOS; Adenomyosis; Other: ovarian cysts; Other: pelvic inflammatory disease
  - Endometriosis; Thyroi

In [67]:
# Standardize and translate "Have you been told you have any of the following?" to English
target_col = 'Have you been told you have any of the following? (select all that apply)'

def standardize_conditions_english(val):
    if pd.isnull(val):
        return None
    
    text = str(val).strip()
    if text.lower() in ['', 'nan']:
        return None
    
    # Translation mappings for "Other:" entries
    translations = {
        'ciste na jajnicima': 'Ovarian cysts',
        'endomentrioza': 'Endometriosis',
        'hronično stanje bola': 'Chronic pain condition',
        'hronichno stanje bola': 'Chronic pain condition',
        'autoimuna bolest (npr. lupus': 'Autoimmune disease (e.g., lupus',
        'reumatoidni artritis': 'Rheumatoid arthritis',
        'upalna bolest karlice': 'Pelvic inflammatory disease',
        'hidradenitis suppurativa': 'Hidradenitis suppurativa',
        'policističan izgled jajnika': 'Polycystic-looking ovaries',
        'ali bez sindroma policističnih jajnika': 'Without polycystic ovary syndrome',
        'cervikobrahijalni sindrom': 'Cervicobrachial syndrome',
        'polip': 'Polyp',
        'ir': 'Insulin resistance',
        'nisam bila na pregledu': "Haven't had examination",
        'yukarıdakilerin hiçbiri': 'None of the above',
        'vjerovatno ms (trenutno u fazi postavljanja dijagnoze)': 'Probable MS (currently in diagnostic phase)',
        'u procesu dijagnoze endo (nije sluzbeno doktor sumnja)': 'In process of endo diagnosis (doctor suspects)',
        'kronik ağrı durumu': 'Chronic pain condition',
        'ovarian cysts': 'Ovarian cysts',
        'chronic pain condition': 'Chronic pain condition',
        'chron': 'Chronic condition',
        'none': 'None of the above',
        'Ništa od navedenog': 'None of the above',
        'none of the above': 'None of the above',
    }
    
    # Split by semicolon and process each part
    parts = [p.strip() for p in text.split(';')]
    translated_parts = []
    
    for part in parts:
        if part.startswith('Other: '):
            other_text = part[7:].strip()  # Remove 'Other: ' prefix
            # Try to translate the other text
            translated = translations.get(other_text, other_text)
            translated_parts.append(translated)
        else:
            # Already standardized labels
            translated_parts.append(part)
    
    # Remove duplicates while preserving order
    unique_parts = []
    seen = set()
    for part in translated_parts:
        if part not in seen:
            unique_parts.append(part)
            seen.add(part)
    
    return '; '.join(unique_parts) if unique_parts else None

if target_col in optimized_df.columns:
    optimized_df[target_col] = optimized_df[target_col].apply(standardize_conditions_english)
    print(f'✓ Standardized {target_col} to English')
    print('\nUnique standardized values:')
    unique_vals = sorted(optimized_df[target_col].dropna().unique())
    for i, val in enumerate(unique_vals, 1):
        print(f'{i}. {val}')
    
    optimized_df.to_csv('../data/optimized_dataset.csv', index=False)
    print(f'\n✓ Updated optimized_df saved to ../data/optimized_dataset.csv')
else:
    print(f'Column NOT found: {target_col}')

✓ Standardized Have you been told you have any of the following? (select all that apply) to English

Unique standardized values:
1. Adenomyosis; Endometriosis
2. Adenomyosis; Fibroids; Autoimmune disease (e.g., lupus; Ovarian cysts; Endometriosis; Chronic pain condition; reumatoidni artritis); Pelvic inflammatory disease
3. Adenomyosis; Fibroids; Endometriosis
4. Adenomyosis; Fibroids; Ovarian cysts; Chronic pain condition
5. Adenomyosis; Ovarian cysts; Endometriosis
6. Adenomyosis; Ovarian cysts; Endometriosis; Hidradenitis suppurativa
7. Adenomyosis; Thyroid disorder; Endometriosis
8. Chronic pain condition
9. Endometriosis
10. Endometriosis; Adenomyosis
11. Endometriosis; Fibroids
12. Endometriosis; IBS; Thyroid disorder; Chronic pain condition
13. Endometriosis; Ovarian cysts
14. Endometriosis; PCOS; Adenomyosis; Ovarian cysts; pelvic inflammatory disease
15. Endometriosis; Thyroid disorder; Chronic pain condition
16. Fibroids
17. Fibroids; Chronic condition; Ovarian cysts; Endomet

In [68]:
# Standardize and translate "Do you have any of these hormone-related features?" to English
target_col = 'Do you have any of these hormone‑related features? (select all that apply)'

def standardize_hormone_features_english(val):
    if pd.isnull(val):
        return None
    
    text = str(val).strip()
    if text.lower() in ['', 'nan']:
        return None
    
    # Translation mappings for "Other:" entries and multilingual terms
    translations = {
        # Bosnian terms
        'gubitak ili prorjeđivanje kose na tjemenu': 'Hair thinning/hair loss',
        'poteškoće s gubitkom kilograma': 'Difficulty losing weight',
        'pretjeran rast dlačica': 'Excess hair growth',
        'tamnjenje kože': 'Skin darkening',
        'bedra': 'thighs',
        'prsa': 'breasts',
        'stomak': 'abdomen',
        'pazuh': 'armpits',
        'drugi pregibi': 'other folds',
        'gornja usna': 'upper lip',
        'bolovi u crijevima': 'stomach pain',
        # Turkish terms
        'kilo vermede güçlük': 'Difficulty losing weight',
        'göğüs': 'chest',
        'karın': 'abdomen',
        'uyluk': 'thighs',
        'üst dudak': 'upper lip',
        # English variants and "None"
        'difficulty losing weight': 'Difficulty losing weight',
        'excess hair growth (chin': 'Excess hair growth (chin',
        'abdomen': 'abdomen',
        'chest': 'chest',
        'thighs': 'thighs',
        'upper lip': 'upper lip',
        'armpits': 'armpits',
        'folds)': 'folds)',
        'none': 'None of the above',
        'none of the above': 'None of the above',
    }
    
    # Split by semicolon and process each part
    parts = [p.strip() for p in text.split(';')]
    translated_parts = []
    
    for part in parts:
        if part.startswith('Other: '):
            other_text = part[7:].strip()  # Remove 'Other: ' prefix
            # Try to translate the other text (case-insensitive)
            translated = translations.get(other_text.lower(), other_text)
            # If it starts with "Other:" prefix again in the result, it means it's a nested structure
            # Just use the other_text as is
            if translated == other_text:
                # Check for partial matches at start of other_text
                for key, val_trans in translations.items():
                    if other_text.lower().startswith(key):
                        translated = val_trans
                        break
            translated_parts.append(translated)
        else:
            # Already standardized labels
            translated_parts.append(part)
    
    # Remove duplicates while preserving order
    unique_parts = []
    seen = set()
    for part in translated_parts:
        if part not in seen:
            unique_parts.append(part)
            seen.add(part)
    
    return '; '.join(unique_parts) if unique_parts else None

if target_col in optimized_df.columns:
    optimized_df[target_col] = optimized_df[target_col].apply(standardize_hormone_features_english)
    print(f'✓ Standardized {target_col} to English')
    print('\nUnique standardized values:')
    unique_vals = sorted(optimized_df[target_col].dropna().unique())
    for i, val in enumerate(unique_vals, 1):
        print(f'{i}. {val}')
    
    optimized_df.to_csv('../data/optimized_dataset.csv', index=False)
    print(f'\n✓ Updated optimized_df saved to ../data/optimized_dataset.csv')
else:
    print(f'Column NOT found: {target_col}')

✓ Standardized Do you have any of these hormone‑related features? (select all that apply) to English

Unique standardized values:
1. Acne/oily skin
2. Acne/oily skin; Difficulty losing weight
3. Acne/oily skin; Difficulty losing weight; Excess hair growth; Skin darkening; thighs; other folds; upper lip; armpits; breasts; abdomen
4. Acne/oily skin; Difficulty losing weight; Excess hair growth; thighs; upper lip; breasts; abdomen
5. Acne/oily skin; Excess hair growth (chin; Difficulty losing weight; Skin darkening; abdomen; chest; other folds; armpits; thighs; upper lip
6. Acne/oily skin; Excess hair growth (chin; Difficulty losing weight; abdomen; chest; thighs; upper lip
7. Acne/oily skin; Excess hair growth (chin; Hair thinning/hair loss; Skin darkening; abdomen; chest; other folds; armpits; thighs; upper lip
8. Acne/oily skin; Excess hair growth (chin; Skin darkening; abdomen; chest; other folds; armpits; thighs; upper lip
9. Acne/oily skin; Excess hair growth (chin; abdomen; chest; 

In [69]:
col_ultrasound_findings = 'Do you have any of these hormone‑related features? (select all that apply)'
if col_ultrasound_findings in optimized_df.columns:
    print(f'Column found: {col_ultrasound_findings}')
    print('Unique values:')
    unique_vals = optimized_df[col_ultrasound_findings].dropna().unique()
    for val in sorted(unique_vals):
        print(f'  - {val}')
else:
    print(f'Column NOT found: {col_ultrasound_findings}')

Column found: Do you have any of these hormone‑related features? (select all that apply)
Unique values:
  - Acne/oily skin
  - Acne/oily skin; Difficulty losing weight
  - Acne/oily skin; Difficulty losing weight; Excess hair growth; Skin darkening; thighs; other folds; upper lip; armpits; breasts; abdomen
  - Acne/oily skin; Difficulty losing weight; Excess hair growth; thighs; upper lip; breasts; abdomen
  - Acne/oily skin; Excess hair growth (chin; Difficulty losing weight; Skin darkening; abdomen; chest; other folds; armpits; thighs; upper lip
  - Acne/oily skin; Excess hair growth (chin; Difficulty losing weight; abdomen; chest; thighs; upper lip
  - Acne/oily skin; Excess hair growth (chin; Hair thinning/hair loss; Skin darkening; abdomen; chest; other folds; armpits; thighs; upper lip
  - Acne/oily skin; Excess hair growth (chin; Skin darkening; abdomen; chest; other folds; armpits; thighs; upper lip
  - Acne/oily skin; Excess hair growth (chin; abdomen; chest; thighs; upper lip

In [70]:
col_ultrasound_findings = 'Have you ever had a pelvic ultrasound? If yes, were you ever told any of these findings? (select all that apply)'
if col_ultrasound_findings in optimized_df.columns:
    print(f'Column found: {col_ultrasound_findings}')
    print('Unique values:')
    unique_vals = optimized_df[col_ultrasound_findings].dropna().unique()
    for val in sorted(unique_vals):
        print(f'  - {val}')
else:
    print(f'Column NOT found: {col_ultrasound_findings}')

Column found: Have you ever had a pelvic ultrasound? If yes, were you ever told any of these findings? (select all that apply)
Unique values:
  - Adenomyosis; Ovarian cyst; Policističan izgled jajnika (opisano kao „policistični“ ili „mnogo malih folikula“); Fluid in the pelvis; Thickened endometrium
  - Adenomyosis; Ovarian cyst; Policističan izgled jajnika (opisano kao „policistični“ ili „mnogo malih folikula“); Thickened endometrium
  - Adenomyosis; Ovarian cyst; Thickened endometrium
  - Adenomyosis; Retrocervikalno se nalaze čvorovi i na sakrouterinim ligamentima
  - Cervical wound (CIN 3)
  - Don't remember; Never had ultrasound
  - Endometrioma/ovarian cyst
  - Endometrioma/ovarian cyst; Adenomyosis
  - Endometrioma/ovarian cyst; Adenomyosis; Ovarian cyst; Policističan izgled jajnika (opisano kao „policistični“ ili „mnogo malih folikula“); Thickened endometrium
  - Endometrioma/ovarian cyst; Adenomyosis; Thickened endometrium
  - Endometrioma/ovarian cyst; Fibroids
  - Endometrio

In [71]:
import unicodedata
# Standardize and translate "Have you ever had a pelvic ultrasound?" to English
target_col = 'Have you ever had a pelvic ultrasound? If yes, were you ever told any of these findings? (select all that apply)'


def normalize_text(s):
    if pd.isnull(s):
        return None

    s = str(s).strip()

    # Normalize Unicode first
    s = unicodedata.normalize('NFKC', s)

    # Standardize quotes/dashes
    replacements = {
        '“': '"',
        '”': '"',
        '„': '"',
        '‟': '"',
        '’': "'",
        '‘': "'",
        '‚': "'",
        '‛': "'",
        '‐': '-',
        '‑': '-',
        '‒': '-',
        '–': '-',
        '—': '-',
        '−': '-',
        '\xa0': ' ',
    }
    for old, new in replacements.items():
        s = s.replace(old, new)

    # Collapse repeated whitespace
    s = re.sub(r'\s+', ' ', s).strip()

    # Remove diacritics for matching only
    s_ascii = unicodedata.normalize('NFKD', s)
    s_ascii = ''.join(ch for ch in s_ascii if not unicodedata.combining(ch))

    return s_ascii.lower().strip()


def standardize_ultrasound_findings_english(val):
    if pd.isnull(val):
        return None

    text = str(val).strip()
    if text.lower() in ['', 'nan']:
        return None

    # Exact normalized mappings
    translations = {
        'policistican izgled jajnika (opisano kao "policisticni" ili "mnogo malih folikula")':
            "Polycystic-looking ovaries (described as 'polycystic' or 'many small follicles')",
        "polikistik gorunumlu overler ('polikistik' veya 'cok sayida kucuk folikul' olarak tanimlandi)":
            "Polycystic-looking ovaries (described as 'polycystic' or 'many small follicles')",
        'polikistik gorunumlu overler ("polikistik" veya "cok sayida kucuk folikul" olarak tanimlandi)':
            "Polycystic-looking ovaries (described as 'polycystic' or 'many small follicles')",
        "polycystic-looking ovaries (described as 'polycystic' or 'many small follicles')":
            "Polycystic-looking ovaries (described as 'polycystic' or 'many small follicles')",
        "polycystic-looking ovaries (described as 'polycystic' or 'many small follicles')":
            "Polycystic-looking ovaries (described as 'polycystic' or 'many small follicles')",

        'retrocervikalno se nalaze cvorovi i na sakrouterinim ligamentima':
            'Nodules on uterosacral ligaments',
        'cvorovi': 'Nodules',
        'retrocervikalno': 'Retrocervical',
        'endometrioza': 'Endometriosis',
        'adenomioza': 'Adenomyosis',

        "i don't remember; i never had an ultrasound": "Don't remember; Never had ultrasound",
        "i don't remember; never had ultrasound": "Don't remember; Never had ultrasound",
        "don't remember": "Don't remember",
        "i don't remember": "Don't remember",
        'never had ultrasound': 'Never had ultrasound',
        "i never had an ultrasound": 'Never had ultrasound',

        'no abnormal findings': 'Everything was fine',
        'everything was fine': 'Everything was fine',
        'none': 'None of the above',
        'none of the above': 'None of the above',
    }

    parts = [p.strip() for p in text.split(';')]
    translated_parts = []

    for part in parts:
        if not part:
            continue

        part_norm = normalize_text(part)
        translated = translations.get(part_norm)

        if translated is None:
            # Partial/fuzzy rule-based matching
            if any(x in part_norm for x in ['policist', 'polikistik', 'polycystic']):
                translated = "Polycystic-looking ovaries (described as 'polycystic' or 'many small follicles')"
            elif 'adenomioz' in part_norm or 'adenomyosis' in part_norm:
                translated = 'Adenomyosis'
            elif 'endometrioza' in part_norm:
                translated = 'Endometriosis'
            elif 'endometrioma' in part_norm:
                translated = 'Endometrioma/ovarian cyst'
            elif 'fibroid' in part_norm or 'miom' in part_norm:
                translated = 'Fibroids'
            elif 'ovarian cyst' in part_norm:
                translated = 'Ovarian cyst'
            elif 'fluid in the pelvis' in part_norm or 'fluid' in part_norm:
                translated = 'Fluid in the pelvis'
            elif 'thickened endometrium' in part_norm or 'zadebljan endometrij' in part_norm:
                translated = 'Thickened endometrium'
            elif 'heart-shaped uterus' in part_norm:
                translated = 'Heart-shaped uterus'
            elif 'cervical wound' in part_norm or 'cin 3' in part_norm:
                translated = 'Cervical wound (CIN 3)'
            elif "don't remember" in part_norm or 'dont remember' in part_norm:
                translated = "Don't remember"
            elif 'never had ultrasound' in part_norm or 'never had an ultrasound' in part_norm:
                translated = 'Never had ultrasound'
            elif part_norm in ['none', 'none of the above']:
                translated = 'None of the above'
            elif part_norm == 'no abnormal findings':
                translated = 'Everything was fine'
            else:
                translated = part.strip()

        translated_parts.append(translated)

    # Remove duplicates while preserving order
    unique_parts = []
    seen = set()
    for part in translated_parts:
        part_key = normalize_text(part)
        if part_key not in seen:
            unique_parts.append(part)
            seen.add(part_key)

    # Optional cleanup:
    # if both "Don't remember" and "Never had ultrasound" exist, combine them
    keys = {normalize_text(x) for x in unique_parts}
    if "don't remember" in keys and 'never had ultrasound' in keys:
        unique_parts = [x for x in unique_parts if normalize_text(x) not in {"don't remember", 'never had ultrasound'}]
        unique_parts.insert(0, "Don't remember; Never had ultrasound")

    return '; '.join(unique_parts) if unique_parts else None


if target_col in optimized_df.columns:
    print(f'✓ Column found: {target_col}')

    optimized_df[target_col] = optimized_df[target_col].apply(standardize_ultrasound_findings_english)
    print(f'\n✓ Standardized {target_col} to English')

    print('\nUnique standardized values:')
    unique_vals = sorted(optimized_df[target_col].dropna().unique())
    for i, val in enumerate(unique_vals, 1):
        print(f'{i}. {val}')

    optimized_df.to_csv('../data/optimized_dataset.csv', index=False)
    print(f'\n✓ Updated optimized_df saved to ../data/optimized_dataset.csv')
else:
    print(f'Column NOT found: {target_col}')

✓ Column found: Have you ever had a pelvic ultrasound? If yes, were you ever told any of these findings? (select all that apply)

✓ Standardized Have you ever had a pelvic ultrasound? If yes, were you ever told any of these findings? (select all that apply) to English

Unique standardized values:
1. Adenomyosis; Nodules on uterosacral ligaments
2. Adenomyosis; Ovarian cyst; Polycystic-looking ovaries (described as 'polycystic' or 'many small follicles'); Fluid in the pelvis; Thickened endometrium
3. Adenomyosis; Ovarian cyst; Polycystic-looking ovaries (described as 'polycystic' or 'many small follicles'); Thickened endometrium
4. Adenomyosis; Ovarian cyst; Thickened endometrium
5. Cervical wound (CIN 3)
6. Don't remember; Never had ultrasound
7. Endometrioma/ovarian cyst
8. Endometrioma/ovarian cyst; Adenomyosis
9. Endometrioma/ovarian cyst; Adenomyosis; Ovarian cyst; Polycystic-looking ovaries (described as 'polycystic' or 'many small follicles'); Thickened endometrium
10. Endometrio

In [72]:
# Load and display unique values from optimized_dataset.csv
df_optimized = pd.read_csv('../data/optimized_dataset.csv')

col_name = 'Have you ever had a pelvic ultrasound? If yes, were you ever told any of these findings? (select all that apply)'

if col_name in df_optimized.columns:
    print(f'✓ Column found: {col_name}\n')
    print(f'Total unique values: {df_optimized[col_name].nunique()}')
    print(f'Total null values: {df_optimized[col_name].isna().sum()}\n')
    print('Unique values:')
    unique_vals = sorted(df_optimized[col_name].dropna().unique())
    for i, val in enumerate(unique_vals, 1):
        print(f'{i}. {val}')
else:
    print(f'Column NOT found in optimized_dataset.csv')

✓ Column found: Have you ever had a pelvic ultrasound? If yes, were you ever told any of these findings? (select all that apply)

Total unique values: 37
Total null values: 1

Unique values:
1. Adenomyosis; Nodules on uterosacral ligaments
2. Adenomyosis; Ovarian cyst; Polycystic-looking ovaries (described as 'polycystic' or 'many small follicles'); Fluid in the pelvis; Thickened endometrium
3. Adenomyosis; Ovarian cyst; Polycystic-looking ovaries (described as 'polycystic' or 'many small follicles'); Thickened endometrium
4. Adenomyosis; Ovarian cyst; Thickened endometrium
5. Cervical wound (CIN 3)
6. Don't remember; Never had ultrasound
7. Endometrioma/ovarian cyst
8. Endometrioma/ovarian cyst; Adenomyosis
9. Endometrioma/ovarian cyst; Adenomyosis; Ovarian cyst; Polycystic-looking ovaries (described as 'polycystic' or 'many small follicles'); Thickened endometrium
10. Endometrioma/ovarian cyst; Adenomyosis; Thickened endometrium
11. Endometrioma/ovarian cyst; Fibroids
12. Endometriom

In [74]:
col_diet_change = 'Have you changed your diet specifically to help with endometriosis / PCOS symptoms? (Select all that apply)'
if col_diet_change in optimized_df.columns:
    print(f'Column found: {col_diet_change}')
    print('Unique values:')
    unique_vals = optimized_df[col_diet_change].dropna().unique()
    for val in sorted(unique_vals):
        print(f'  - {val}')
else:
    print(f'Column NOT found: {col_diet_change}')

Column found: Have you changed your diet specifically to help with endometriosis / PCOS symptoms? (Select all that apply)
Unique values:
  - Higher protein diet
  - Higher protein diet; Keto diet; Reduced sugar/sweets; Reduced processed/fast food; Reduced gluten; Reduced dairy; Vegetarian/vegan; Intermittent fasting/time-restricted eating; Other: Anti‑inflammatory diet (for example focusing on whole foods; Other: fruits; Other: healthy fats); Other: sweets; Other: vegetables
  - Higher protein diet; Reduced sugar/sweets; Other: Anti‑inflammatory diet (for example focusing on whole foods; Other: fruits; Other: healthy fats); Other: sweets; Other: vegetables
  - Higher protein diet; Reduced sugar/sweets; Reduced processed/fast food; Other: Anti‑inflammatory diet (for example focusing on whole foods; Other: fruits; Other: healthy fats); Other: sweets; Other: vegetables
  - Intermittent fasting/time-restricted eating
  - Intermittent fasting/time-restricted eating; Other: Antiinflamatorna 

In [79]:
import re
import unicodedata
import pandas as pd

target_col = "Have you changed your diet specifically to help with endometriosis / PCOS symptoms? (Select all that apply)"


def normalize_text(s):
    if pd.isnull(s):
        return None

    s = str(s).strip()

    s = unicodedata.normalize('NFKC', s)

    replacements = {
        '“': '"', '”': '"', '„': '"', '‟': '"',
        '’': "'", '‘': "'", '‚': "'", '‛': "'",
        '‐': '-', '‑': '-', '‒': '-', '–': '-', '—': '-', '−': '-',
        '\xa0': ' ',
    }
    for old, new in replacements.items():
        s = s.replace(old, new)

    s = re.sub(r'\s+', ' ', s).strip()

    s_ascii = unicodedata.normalize('NFKD', s)
    s_ascii = ''.join(ch for ch in s_ascii if not unicodedata.combining(ch))

    # extra manual normalization for Turkish/Balkan chars that may survive
    char_map = str.maketrans({
        'ı': 'i',
        'İ': 'i',
        'ş': 's',
        'Ş': 's',
        'ğ': 'g',
        'Ğ': 'g',
        'č': 'c',
        'Č': 'c',
        'ć': 'c',
        'Ć': 'c',
        'ž': 'z',
        'Ž': 'z',
        'đ': 'd',
        'Đ': 'd',
    })
    s_ascii = s_ascii.translate(char_map)

    return s_ascii.lower().strip()


BASE_OPTIONS = {
    "higher protein diet": "Higher protein diet",
    "keto diet": "Keto diet",
    "lower-carb diet": "Lower-carb diet",
    "low-carb diet": "Lower-carb diet",
    "reduced sugar/sweets": "Reduced sugar/sweets",
    "reduced processed/fast food": "Reduced processed/fast food",
    "reduced gluten": "Reduced gluten",
    "reduced dairy": "Reduced dairy",
    "vegetarian/vegan": "Vegetarian/vegan",
    "intermittent fasting/time-restricted eating": "Intermittent fasting/time-restricted eating",
    "none": "None",
}


def clean_other_prefix(part):
    return re.sub(r'^other:\s*', '', str(part).strip(), flags=re.IGNORECASE).strip()


def map_other_fragment(part, norm):
    cleaned = clean_other_prefix(part)

    # High protein
    if (
        "visokoproteinska ishrana" in norm
        or "vecim unosom proteina" in norm
        or "vecim unosom proteina" in norm
        or "yüksek protein" in norm
        or "yuksek protein" in norm
    ):
        return "Higher protein diet"

    # Low carb
    if (
        "smanjenim unosom ugljikohidrata" in norm
        or "low-carb ishrana" in norm
        or "low-carb diyet" in norm
        or norm == "low-carb diet"
    ):
        return "Lower-carb diet"

    # Reduced gluten
    if "smanjen unos glutena" in norm or "gluteni azaltmak" in norm:
        return "Reduced gluten"

    # Reduced dairy
    if (
        "smanjen unos mlijecnih proizvoda" in norm
        or "smanjen unos mlijecnih" in norm
        or "smanjen unos mliječnih proizvoda" in str(part).lower()
    ):
        return "Reduced dairy"

    # Reduced processed / fast food
    if (
        "smanjen unos preradjene hrane" in norm
        or "smanjen unos preradjene" in norm
        or "islenmis gidalar" in norm
        or "brze hrane" in norm
    ):
        return "Reduced processed/fast food"

    # Reduced sugar / sweets
    if (
        "smanjen unos secera" in norm
        or "smanjen unos slatkisa" in norm
        or "slatkisa" in norm
        or norm == "sweets"
        or "tatlilari azaltmak" in norm
        or norm == "seker"
        or "reduced sugar" in norm
        or "reduced sweets" in norm
    ):
        return "Reduced sugar/sweets"

    # Intermittent fasting
    if (
        "zaman kisitli yeme" in norm
        or "intermittent fasting" in norm
        or "time-restricted eating" in norm
    ):
        return "Intermittent fasting/time-restricted eating"

    # Anti-inflammatory
    if (
        "antiinflamatorna" in norm
        or "anti-inflamatuar" in norm
        or "anti-inflammatory" in norm
    ):
        return "Anti-inflammatory diet (focus on whole foods)"

    # None / no change
    if (
        "nijedna" in norm
        or "hicbir degisiklik" in norm
        or "i have not changed my diet" in norm
        or "nisam mijenjala ishranu radi ovih simptoma" in norm
        or "nisam menja" in norm
    ):
        return "None"

    # Sentence: did not try long-term, but reduced sugar/caffeine/processed food
    if "nisam dugorocno probala promenu ishrane" in norm:
        if "secera" in norm or "sugar" in norm:
            return "Reduced sugar/sweets"
        return "No long-term diet change"

    # Vegetables / fruit / fats
    if "povrce" in norm or "sebze" in norm or norm == "vegetables":
        return "More vegetables"

    if "voce" in norm or "meyve" in norm or norm == "fruits":
        return "More fruits"

    if "zdrave masti" in norm or "saglikli yaglar" in norm or "healthy fats" in norm:
        return "More healthy fats"

    # Coffee / caffeine
    if "coffee" in norm or "kofein" in norm or "caffeine" in norm:
        if "stopped" in norm or "prestala" in norm or "ne pijem" in norm:
            return "Stopped coffee/caffeine"
        return "Reduced coffee/caffeine"

    # Magnesium / bananas
    if "magnezij" in norm or "magnesium" in norm:
        return "More magnesium-rich foods"

    if "banane" in norm and "ublazavanje bolova" in norm:
        return "More bananas for pain relief"

    # I don't have it
    if "i dont have it" in norm or "i don't have it" in norm:
        return "I do not have endometriosis/PCOS"

    # Vegetarian unrelated
    if "i am a vegeterian not related" in norm or "i am a vegetarian not related" in norm:
        return "Vegetarian (not related to symptoms)"

    # If no mapping found, return cleaned fragment
    return cleaned if cleaned else part.strip()


def standardize_diet_changes_english(val):
    if pd.isnull(val):
        return None

    text = str(val).strip()
    if text.lower() in ['', 'nan']:
        return None

    parts = [p.strip() for p in text.split(';')]
    translated_parts = []

    for part in parts:
        if not part:
            continue

        norm = normalize_text(part)

        # Exact base option match
        base_match = None
        for base_norm, base_label in BASE_OPTIONS.items():
            if norm == base_norm:
                base_match = base_label
                break

        if base_match:
            translated_parts.append(base_match)
            continue

        # Other:
        if norm.startswith("other:"):
            translated_parts.append(map_other_fragment(part, norm))
            continue

        # Fuzzy matching for regular values / stray leftovers
        if "higher protein" in norm:
            translated_parts.append("Higher protein diet")

        elif "keto" in norm:
            translated_parts.append("Keto diet")

        elif "lower-carb" in norm or "low-carb" in norm:
            translated_parts.append("Lower-carb diet")

        elif (
            "reduced sugar" in norm
            or "reduced sweets" in norm
            or ("reduced" in norm and ("sugar" in norm or "sweets" in norm))
            or "slatkisa" in norm
            or norm == "sweets"
        ):
            translated_parts.append("Reduced sugar/sweets")

        elif (
            "processed/fast food" in norm
            or ("processed" in norm and "food" in norm)
            or "islenmis gidalar" in norm
        ):
            translated_parts.append("Reduced processed/fast food")

        elif "reduced gluten" in norm:
            translated_parts.append("Reduced gluten")

        elif "reduced dairy" in norm:
            translated_parts.append("Reduced dairy")

        elif "vegetarian/vegan" in norm:
            translated_parts.append("Vegetarian/vegan")

        elif "intermittent fasting" in norm or "time-restricted eating" in norm:
            translated_parts.append("Intermittent fasting/time-restricted eating")

        elif norm == "none" or "hicbir degisiklik" in norm or "nisam mijenjala ishranu" in norm:
            translated_parts.append("None")

        elif "anti-inflammatory" in norm:
            translated_parts.append("Anti-inflammatory diet (focus on whole foods)")

        elif "povrce" in norm or "sebze" in norm:
            translated_parts.append("More vegetables")

        elif "voce" in norm or "meyve" in norm:
            translated_parts.append("More fruits")

        elif "saglikli yaglar" in norm or "healthy fats" in norm:
            translated_parts.append("More healthy fats")

        elif "banane" in norm and "ublazavanje bolova" in norm:
            translated_parts.append("More bananas for pain relief")

        elif "nisam dugorocno probala promenu ishrane" in norm:
            if "secera" in norm:
                translated_parts.append("Reduced sugar/sweets")
            else:
                translated_parts.append("No long-term diet change")

        else:
            translated_parts.append(part.strip())

    # Deduplicate while preserving order
    unique_parts = []
    seen = set()
    for p in translated_parts:
        key = normalize_text(p)
        if key not in seen:
            unique_parts.append(p)
            seen.add(key)

    # Cleanup rules
    keys = {normalize_text(x) for x in unique_parts}

    # If None is present with any "no change" variants, keep only None
    if "none" in keys:
        unique_parts = [x for x in unique_parts if normalize_text(x) not in {
            "nisam mijenjala ishranu radi ovih simptoma",
            "i have not changed my diet for these symptoms"
        }]

    # If Reduced sugar/sweets exists, remove stray sweets leftovers
    cleaned_parts = []
    seen2 = set()
    for p in unique_parts:
        k = normalize_text(p)
        if k in {"slatkisa", "sweets"} and "reduced sugar/sweets" in keys:
            continue
        if k not in seen2:
            cleaned_parts.append(p)
            seen2.add(k)

    return '; '.join(cleaned_parts) if cleaned_parts else None


if target_col in optimized_df.columns:
    print(f"✓ Column found: {target_col}")

    optimized_df[target_col] = optimized_df[target_col].apply(standardize_diet_changes_english)
    print(f"\n✓ Standardized {target_col} to English")

    print("\nUnique standardized values:")
    unique_vals = sorted(optimized_df[target_col].dropna().unique())
    for i, val in enumerate(unique_vals, 1):
        print(f"{i}. {val}")

    optimized_df.to_csv("../data/optimized_dataset.csv", index=False)
    print("\n✓ Updated optimized_df saved to ../data/optimized_dataset.csv")
else:
    print(f"Column NOT found: {target_col}")

✓ Column found: Have you changed your diet specifically to help with endometriosis / PCOS symptoms? (Select all that apply)

✓ Standardized Have you changed your diet specifically to help with endometriosis / PCOS symptoms? (Select all that apply) to English

Unique standardized values:
1. Anti-inflammatory diet (focus on whole foods); Higher protein diet; Reduced dairy; Reduced processed/fast food; Reduced sugar/sweets; More vegetables; More fruits; More healthy fats
2. Anti-inflammatory diet (focus on whole foods); Lower-carb diet; Higher protein diet; Reduced dairy; Reduced processed/fast food; Reduced sugar/sweets; More vegetables; More fruits; More healthy fats
3. Anti-inflammatory diet (focus on whole foods); Lower-carb diet; Higher protein diet; Reduced gluten; Reduced dairy; Reduced processed/fast food; Reduced sugar/sweets; More vegetables; More fruits; More healthy fats
4. Anti-inflammatory diet (focus on whole foods); Lower-carb diet; Higher protein diet; Reduced processed/f

In [80]:

col_activity = 'Which types of physical activity have you used specifically to help your symptoms? (Select all that apply)'
if col_activity in optimized_df.columns:
    print(f'Column found: {col_activity}')
    print('Unique values:')
    unique_vals = optimized_df[col_activity].dropna().unique()
    for val in sorted(unique_vals):
        print(f'  - {val}')
else:
    print(f'Column NOT found: {col_activity}')

Column found: Which types of physical activity have you used specifically to help your symptoms? (Select all that apply)
Unique values:
  - Gentle stretching/mobility
  - Gentle stretching/mobility; Pelvic floor/physiotherapy
  - Gentle stretching/mobility; Running/jogging; Cardio
  - Gentle stretching/mobility; Running/jogging; Weight training
  - Gentle stretching/mobility; Walking
  - Gentle stretching/mobility; Walking; Cardio
  - Gentle stretching/mobility; Walking; Other: Vježbe za karlično dno; Other: fizioterapeutske vježbe
  - Gentle stretching/mobility; Walking; Running/jogging; Other: Grupni treninzi (na primjer aerobik; Other: Zumba i slično); Other: ples
  - Gentle stretching/mobility; Walking; Running/jogging; Weight training
  - Gentle stretching/mobility; Walking; Running/jogging; Weight training; Cardio
  - Gentle stretching/mobility; Walking; Swimming/water exercise
  - Gentle stretching/mobility; Walking; Weight training
  - Gentle stretching/mobility; Walking; Weigh

In [86]:
import re
import unicodedata
import pandas as pd

target_col = "Which types of physical activity have you used specifically to help your symptoms? (Select all that apply)"


def normalize_text(s):
    if pd.isnull(s):
        return None

    s = str(s).strip()

    s = unicodedata.normalize('NFKC', s)

    replacements = {
        '“': '"', '”': '"', '„': '"', '‟': '"',
        '’': "'", '‘': "'", '‚': "'", '‛': "'",
        '‐': '-', '‑': '-', '‒': '-', '–': '-', '—': '-', '−': '-',
        '\xa0': ' ',
    }
    for old, new in replacements.items():
        s = s.replace(old, new)

    s = re.sub(r'\s+', ' ', s).strip()

    s_ascii = unicodedata.normalize('NFKD', s)
    s_ascii = ''.join(ch for ch in s_ascii if not unicodedata.combining(ch))

    char_map = str.maketrans({
        'ı': 'i', 'İ': 'i',
        'ş': 's', 'Ş': 's',
        'ğ': 'g', 'Ğ': 'g',
        'č': 'c', 'Č': 'c',
        'ć': 'c', 'Ć': 'c',
        'ž': 'z', 'Ž': 'z',
        'đ': 'd', 'Đ': 'd',
    })
    s_ascii = s_ascii.translate(char_map)

    return s_ascii.lower().strip()


BASE_OPTIONS = {
    "gentle stretching/mobility": "Gentle stretching/mobility",
    "yoga/pilates": "Yoga/Pilates",
    "walking": "Walking",
    "running/jogging": "Running/jogging",
    "weight training": "Weight training",
    "cardio": "Cardio",
    "swimming/water exercise": "Swimming/water exercise",
    "pelvic floor/physiotherapy": "Pelvic floor/physiotherapy",
    "group classes (for example aerobics, zumba, dance)": "Group classes (for example aerobics, Zumba, dance)",
    "none": "None",
}


def clean_other_prefix(part):
    return re.sub(r'^other:\s*', '', str(part).strip(), flags=re.IGNORECASE).strip()


def map_other_fragment(part, norm):
    cleaned = clean_other_prefix(part)

    # Pelvic floor / physiotherapy
    if "vjezbe za karlicno dno" in norm or "vjezbe za karlicno" in norm:
        return "Pelvic floor/physiotherapy"
    if "fizioterapeutske vjezbe" in norm or "fizioterap" in norm:
        return "Pelvic floor/physiotherapy"

    # Group classes
    if "grupni treninzi" in norm or "aerobik" in norm or "zumba" in norm or "ples" in norm:
        return "Group classes (for example aerobics, Zumba, dance)"
    if "group classes" in norm or "aerobics" in norm or "dance" in norm:
        return "Group classes (for example aerobics, Zumba, dance)"

    # None / no activity used
    if "nijedna" in norm and "nisam koristila fizicku aktivnost" in norm:
        return "None"
    if "hicbirini denemedim" in norm:
        return "None"
    if "i have not used physical activity as a remedy" in norm:
        return "None"

    return cleaned if cleaned else part.strip()


def standardize_physical_activity_english(val):
    if pd.isnull(val):
        return None

    text = str(val).strip()
    if text.lower() in ['', 'nan']:
        return None

    norm_full = normalize_text(text)

    # HARD OVERRIDE: if whole answer is that Bosnian "no activity" sentence → map entire row to None
    if (
        "nijedna" in norm_full
        and "nisam koristila fizicku aktivnost kao sredstvo za ublazavanje simptoma" in norm_full
    ):
        return "None"

    parts = [p.strip() for p in text.split(';')]
    translated_parts = []

    for part in parts:
        if not part:
            continue


        norm = normalize_text(part)

        # 1) Exact base option match
        base_match = None
        for base_norm, base_label in BASE_OPTIONS.items():
            if norm == base_norm:
                base_match = base_label
                break
        if base_match:
            translated_parts.append(base_match)
            continue

        # 2) Handle "Other:"
        if norm.startswith("other:"):
            mapped = map_other_fragment(part, norm)
            translated_parts.append(mapped)
            continue

        # 3) Fuzzy/partial matches

        if "gentle stretching" in norm or "mobility" in norm:
            translated_parts.append("Gentle stretching/mobility")

        elif "yoga" in norm or "pilates" in norm:
            translated_parts.append("Yoga/Pilates")

        elif "walking" in norm:
            translated_parts.append("Walking")

        elif "running" in norm or "jogging" in norm:
            translated_parts.append("Running/jogging")

        elif "weight training" in norm or "weights" in norm:
            translated_parts.append("Weight training")

        elif "cardio" in norm:
            translated_parts.append("Cardio")

        elif "swimming" in norm or "water exercise" in norm:
            translated_parts.append("Swimming/water exercise")

        elif "pelvic floor" in norm or "physiotherapy" in norm:
            translated_parts.append("Pelvic floor/physiotherapy")

        elif (
            "group classes" in norm
            or "aerobics" in norm
            or "zumba" in norm
            or "dance" in norm
        ):
            translated_parts.append("Group classes (for example aerobics, Zumba, dance)")

        elif norm == "none":
            translated_parts.append("None")

        else:
            translated_parts.append(part.strip())

    # Deduplicate while preserving order
    unique_parts = []
    seen = set()
    for p in translated_parts:
        key = normalize_text(p)
        if key not in seen:
            unique_parts.append(p)
            seen.add(key)

    # Cleanup: drop comment-like fragments such as "etc.)"
    cleaned_parts = []
    for p in unique_parts:
        k = normalize_text(p)

        # drop "etc.)"
        if k in {"etc", "etc)"}:
            continue

        cleaned_parts.append(p)

    # Final: if both None and explanations exist, keep just "None"
    if any(normalize_text(x) == "none" for x in cleaned_parts):
        cleaned_parts = ["None"] if "None" in cleaned_parts else cleaned_parts

    return '; '.join(cleaned_parts) if cleaned_parts else None


if target_col in optimized_df.columns:
    print(f"✓ Column found: {target_col}")

    optimized_df[target_col] = optimized_df[target_col].apply(standardize_physical_activity_english)
    print(f"\n✓ Standardized {target_col} to English")

    print("\nUnique standardized values:")
    unique_vals = sorted(optimized_df[target_col].dropna().unique())
    for i, val in enumerate(unique_vals, 1):
        print(f"{i}. {val}")

    optimized_df.to_csv("../data/optimized_dataset.csv", index=False)
    print("\n✓ Updated optimized_df saved to ../data/optimized_dataset.csv")
else:
    print(f"Column NOT found: {target_col}")

✓ Column found: Which types of physical activity have you used specifically to help your symptoms? (Select all that apply)

✓ Standardized Which types of physical activity have you used specifically to help your symptoms? (Select all that apply) to English

Unique standardized values:
1. Gentle stretching/mobility
2. Gentle stretching/mobility; Pelvic floor/physiotherapy
3. Gentle stretching/mobility; Running/jogging; Cardio
4. Gentle stretching/mobility; Running/jogging; Weight training
5. Gentle stretching/mobility; Walking
6. Gentle stretching/mobility; Walking; Cardio
7. Gentle stretching/mobility; Walking; Pelvic floor/physiotherapy
8. Gentle stretching/mobility; Walking; Running/jogging; Group classes (for example aerobics, Zumba, dance)
9. Gentle stretching/mobility; Walking; Running/jogging; Weight training
10. Gentle stretching/mobility; Walking; Running/jogging; Weight training; Cardio
11. Gentle stretching/mobility; Walking; Swimming/water exercise
12. Gentle stretching/mobi

In [93]:

col_remedies_tried = 'Which types of remedies or strategies have you tried for your symptoms? (Select all that apply)\n'

if col_remedies_tried in optimized_df.columns:
    print(f'Column found: {col_remedies_tried}')
    print('Unique values:')
    unique_vals = optimized_df[col_remedies_tried].dropna().unique()
    for val in sorted(unique_vals):
        print(f'  - {val}')
else:
    print(f'Column NOT found: {col_remedies_tried}')

Column found: Which types of remedies or strategies have you tried for your symptoms? (Select all that apply)

Unique values:
  - Heat therapy; Diet changes; Other: Pain medication (for example ibuprofen; Other: hot water bottle; Other: naproxen; Other: paracetamol)
  - Heat therapy; Herbal teas/herbs; Other: Hormonalni tretmani (kontracepcija ili drugi propisani hormoni); Other: Lijekovi protiv bolova (na primjer ibuprofen; Other: boca sa toplom vodom; Other: naproksen; Other: paracetamol); Other: tople kupke)
  - Heat therapy; Herbal teas/herbs; Other: Hormonalni tretmani (kontracepcija ili drugi propisani hormoni); Other: boca sa toplom vodom; Other: tople kupke)
  - Heat therapy; Herbal teas/herbs; Other: Lijekovi protiv bolova (na primjer ibuprofen; Other: boca sa toplom vodom; Other: naproksen; Other: paracetamol); Other: tople kupke)
  - Heat therapy; Herbal teas/herbs; Other: More intense physical activity (running; Other: gym; Other: hot water bottle; Other: sports)
  - Heat t

In [95]:
import re
import unicodedata
import pandas as pd

target_col = "Which types of remedies or strategies have you tried for your symptoms? (Select all that apply)\n"


def normalize_text(s):
    if pd.isnull(s):
        return None

    s = str(s).strip()

    # Normalize Unicode
    s = unicodedata.normalize('NFKC', s)

    # Standardize quotes/dashes and non‑breaking spaces
    replacements = {
        '“': '"', '”': '"', '„': '"', '‟': '"',
        '’': "'", '‘': "'", '‚': "'", '‛': "'",
        '‐': '-', '‑': '-', '‒': '-', '–': '-', '—': '-', '−': '-',
        '\xa0': ' ',
    }
    for old, new in replacements.items():
        s = s.replace(old, new)

    # Collapse whitespace
    s = re.sub(r'\s+', ' ', s).strip()

    # Remove diacritics
    s_ascii = unicodedata.normalize('NFKD', s)
    s_ascii = ''.join(ch for ch in s_ascii if not unicodedata.combining(ch))

    # Extra manual normalization for Balkan/Turkish chars
    char_map = str.maketrans({
        'ı': 'i', 'İ': 'i',
        'ş': 's', 'Ş': 's',
        'ğ': 'g', 'Ğ': 'g',
        'č': 'c', 'Č': 'c',
        'ć': 'c', 'Ć': 'c',
        'ž': 'z', 'Ž': 'z',
        'đ': 'd', 'Đ': 'd',
    })
    s_ascii = s_ascii.translate(char_map)

    return s_ascii.lower().strip()


# Canonical base options you want in the final column
BASE_OPTIONS = {
    "heat therapy": "Heat therapy",
    "herbal teas/herbs": "Herbal teas/herbs",
    "diet changes": "Diet changes",
    "supplements": "Supplements",
    "stretching/mobility": "Stretching/mobility",
    "yoga/pilates": "Yoga/Pilates",
    "walking": "Walking",
    "more intense physical activity (running, gym, sports)": "More intense physical activity (running, gym, sports)",
    "light physical activity (for example yoga)": "Light physical activity (for example yoga)",
    "pain medication (for example ibuprofen, naproxen, paracetamol)": "Pain medication (for example ibuprofen, naproxen, paracetamol)",
    "hot water bottle / heat pack": "Hot water bottle / heat pack",
    "warm baths": "Warm baths",
    "hormonal treatments (contraception or other prescribed hormones)": "Hormonal treatments (contraception or other prescribed hormones)",
    "surgery": "Surgery",
    "alternative medicine": "Alternative medicine",
    "metformin": "Metformin",
    "none": "None",
}


def clean_other_prefix(part):
    return re.sub(r'^other:\s*', '', str(part).strip(), flags=re.IGNORECASE).strip()


def map_other_fragment(part, norm):
    """
    Map 'Other:' fragments in Bosnian/Turkish/English to base options.
    `part` is original substring, `norm` is normalized lower-ascii.
    """

    cleaned = clean_other_prefix(part)

    # --- Hormonal treatments ---
    if (
        "hormonalni tretmani" in norm
        or "hormonal tedaviler" in norm
        or "kontracepcija" in norm
        or "dogum kontrolu" in norm
        or "prescribed hormon" in norm
    ):
        return "Hormonal treatments (contraception or other prescribed hormones)"

    # --- Pain medication ---
    if (
        "lijekovi protiv bolova" in norm
        or "pain medication" in norm
        or "ibuprofen" in norm
        or "naproksen" in norm
        or "naproxen" in norm
        or "paracetamol" in norm
        or "parasetamol" in norm
    ):
        return "Pain medication (for example ibuprofen, naproxen, paracetamol)"

    # --- Hot water bottle / heat pack ---
    if (
        "boca sa toplom vodom" in norm
        or "sicak su torbasi" in norm
        or "sicak su torbasi" in norm
        or "hot water bottle" in norm
        or "isi uygulamasi" in norm
    ):
        return "Hot water bottle / heat pack"

    # --- Warm baths ---
    if "tople kupke" in norm or "sicak banyo" in norm or "warm bath" in norm:
        return "Warm baths"

    # --- More intense physical activity ---
    if (
        "intenzivnija fizicka aktivnost" in norm
        or "more intense physical activity" in norm
        or "running" in norm and "sport" in norm
        or "teretana" in norm
        or "gym" in norm
        or "sports" in norm
    ):
        return "More intense physical activity (running, gym, sports)"

    # --- Light physical activity (yoga etc.) ---
    if "lagana fizicka aktivnost" in norm and "joga" in norm:
        return "Light physical activity (for example yoga)"

    # --- Supplements / vitamins ---
    if "vitamini" in norm or "vitaminler" in norm or "vitamins" in norm:
        return "Supplements"

    # --- Alternative medicine ---
    if "alternativa medicina" in norm or "bitkisel" in norm or "sifali bitkiler" in norm:
        # these often overlap with herbal teas, but keep as separate "Alternative medicine"
        return "Alternative medicine"

    # --- Surgery ---
    if "operacija" in norm or "surgery" in norm:
        return "Surgery"

    # --- Metformin ---
    if "metformin" in norm:
        return "Metformin"

    # --- None / no remedies tried ---
    if (
        "nisam isprobala nikakva rjesenja" in norm
        or "nisam isprobala nikakva rjesenja niti strategije" in norm
        or "i have not tried any remedies or strategies" in norm
    ):
        return "None"

    # If nothing matched, return cleaned raw text
    return cleaned if cleaned else part.strip()


def is_none_full_answer(norm_full):
    # Full-answer override to map to None
    if "nisam isprobala nikakva rjesenja" in norm_full:
        return True
    if "i have not tried any remedies or strategies" in norm_full:
        return True
    if norm_full == "none":
        return True
    return False


def standardize_remedies_english(val):
    if pd.isnull(val):
        return None

    text = str(val).strip()
    if text.lower() in ['', 'nan']:
        return None

    norm_full = normalize_text(text)
    if is_none_full_answer(norm_full):
        return "None"

    parts = [p.strip() for p in text.split(';')]
    translated_parts = []

    for part in parts:
        if not part:
            continue

        norm = normalize_text(part)

        # 1) Exact base option match
        base_match = None
        for base_norm, base_label in BASE_OPTIONS.items():
            if norm == base_norm:
                base_match = base_label
                break
        if base_match:
            translated_parts.append(base_match)
            continue

        # 2) Handle "Other:"
        if norm.startswith("other:"):
            mapped = map_other_fragment(part, norm)
            translated_parts.append(mapped)
            continue

        # 3) Fuzzy / partial matches for non-Other parts

        # Heat therapy
        if "heat therapy" in norm:
            translated_parts.append("Heat therapy")

        # Herbal teas / herbs
        elif "herbal teas/herbs" in norm or ("herbal" in norm and "tea" in norm):
            translated_parts.append("Herbal teas/herbs")

        # Diet changes
        elif "diet changes" in norm:
            translated_parts.append("Diet changes")

        # Supplements
        elif "supplements" in norm:
            translated_parts.append("Supplements")

        # Stretching / mobility
        elif "stretching/mobility" in norm or ("stretching" in norm and "mobility" in norm):
            translated_parts.append("Stretching/mobility")

        # Yoga / Pilates
        elif "yoga/pilates" in norm or "yoga" in norm or "pilates" in norm:
            translated_parts.append("Yoga/Pilates")

        # Walking
        elif "walking" in norm:
            translated_parts.append("Walking")

        # None (already normalized)
        elif norm == "none":
            translated_parts.append("None")

        else:
            translated_parts.append(part.strip())

    # Deduplicate while preserving order
    unique_parts = []
    seen = set()
    for p in translated_parts:
        key = normalize_text(p)
        if key not in seen:
            unique_parts.append(p)
            seen.add(key)

    # Cleanup: drop obvious trailing technical fragments if any (like stray ')')
    cleaned_parts = []
    for p in unique_parts:
        k = normalize_text(p)
        if k in {")", ")", "etc", "etc)"}:
            continue
        cleaned_parts.append(p)

    # Final safety: if "None" is present with any others, keep just "None"
    if any(normalize_text(x) == "none" for x in cleaned_parts):
        cleaned_parts = ["None"]

    return '; '.join(cleaned_parts) if cleaned_parts else None


if target_col in optimized_df.columns:
    print(f"✓ Column found: {target_col}")

    optimized_df[target_col] = optimized_df[target_col].apply(standardize_remedies_english)
    print(f"\n✓ Standardized {target_col} to English")

    print("\nUnique standardized values:")
    unique_vals = sorted(optimized_df[target_col].dropna().unique())
    for i, val in enumerate(unique_vals, 1):
        print(f"{i}. {val}")

    optimized_df.to_csv("../data/optimized_dataset.csv", index=False)
    print("\n✓ Updated optimized_df saved to ../data/optimized_dataset.csv")
else:
    print(f"Column NOT found: {target_col}")

✓ Column found: Which types of remedies or strategies have you tried for your symptoms? (Select all that apply)


✓ Standardized Which types of remedies or strategies have you tried for your symptoms? (Select all that apply)
 to English

Unique standardized values:
1. Heat therapy; Alternative medicine; Pain medication (for example ibuprofen, naproxen, paracetamol); Hot water bottle / heat pack; Warm baths
2. Heat therapy; Diet changes; Pain medication (for example ibuprofen, naproxen, paracetamol); Hot water bottle / heat pack
3. Heat therapy; Herbal teas/herbs; Hormonal treatments (contraception or other prescribed hormones); Hot water bottle / heat pack; Warm baths
4. Heat therapy; Herbal teas/herbs; Hormonal treatments (contraception or other prescribed hormones); Pain medication (for example ibuprofen, naproxen, paracetamol); Hot water bottle / heat pack; Warm baths
5. Heat therapy; Herbal teas/herbs; Hot water bottle / heat pack; Warm baths
6. Heat therapy; Herbal teas/herbs; Mor